In [ ]:
# Repository paths; inputs are read-only, new files stay in results/.
from pathlib import Path
import os
import sys
_start = Path(os.environ.get("SLC7A5_REPO_ROOT", str(Path.cwd()))).resolve()
_repo = next((p for p in [_start, *_start.parents] if (p / "slc7a5_paths.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the repository or set SLC7A5_REPO_ROOT.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from slc7a5_paths import artifact_path, dataset_path, input_path, output_path, output_dir, setup_notebook
setup_notebook('hwangbj')


# HwangBJ: MASL1–4 간 핵 및 MASL/MASH PBMC / Harmony 분석

RDS **4개는 간 핵 counts + metadata, PBMC counts + metadata의 두 쌍**입니다.
간 핵은 MASL1–4, PBMC는 MASL1–4 및 MASH1–4를 선택하여 각 검체를 별도로 분석합니다.
`MASL1–4`는 donor ID이며 질병의 1–4단계가 아닙니다.

각 데이터에서 **QC → batch-aware HVG → PCA → donor Harmony → neighbors/UMAP/Leiden →
myeloid 분리 → myeloid PCA/Harmony 재실행 → monocyte 후보 → SLC7A5 표현형 확인** 순서로 진행합니다.

- 간 핵과 PBMC는 검체 종류가 다르므로 각각 처리하고 별도 폴더에 저장합니다.
- Harmony 보정값 `X_pca_harmony`를 클러스터링에 사용합니다. 원본 UMI와 log-normalized 발현은 유지합니다.
- 전체 세포 및 myeloid에서 보정 전/후 donor UMAP을 저장합니다. donor 간 차이에는 생물학적 차이도 포함되므로 marker와 donor 기여도를 함께 확인합니다.
- 자동 주석은 검토용 후보입니다. 기존 Figure 2의 `mono5/Special_monocyte` 번호를 새 cluster에 적용하지 않습니다.
- 간 핵의 high는 탐색적 발현 gate, PBMC의 high는 참조 DEG matching으로 선택한 cluster입니다. 기존 Special_monocyte와 동일한 세포형임을 확정하지 않습니다.
- PBMC는 MASL 4명/MASH 4명의 donor별 비율을 비교합니다. Healthy는 없으며 cell-level P 값으로 질환군 차이를 검정하지 않습니다.

환경은 저장소의 `environment/requirements-python.txt`와 `docs/hwangbj.md`를 참고합니다.
RDS 변환에는 같은 폴더의 `HwangBJ_export_counts.R`와 Matrix 패키지가 있는 Rscript가 필요합니다.
Harmony는 현재 Python 환경에 설치된 harmonypy 0.0.10을 사용합니다.
[Scanpy Harmony 문서](https://scanpy.readthedocs.io/en/stable/generated/scanpy.external.pp.harmony_integrate.html)


PBMC 주석은 기존 scRNA-seq SLC7A5-high monocyte DEG와 통합 코호트 myeloid cluster의 donor-balanced 효과를 비교한 참조 matching을 사용합니다. 기존 expression-q75 정의는 보조 컬럼에 보존합니다.

In [ ]:
import os
from pathlib import Path
for key, value in {
    'MPLCONFIGDIR': '/tmp/hwangbj-mpl', 'XDG_CACHE_HOME': '/tmp/hwangbj-cache',
    'NUMBA_CACHE_DIR': '/tmp/hwangbj-numba', 'NUMBA_NUM_THREADS': '4',
    'OPENBLAS_NUM_THREADS': '4', 'OMP_NUM_THREADS': '4',
}.items():
    os.environ.setdefault(key, value)

import json, shutil, subprocess, hashlib, platform, sys, importlib.util
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

PROJECT = Path(output_dir('02_SLC7A5_mono_Journal'))
HWANG_SCRIPTS = _repo / 'scripts/hwangbj'
RDS_INPUTS = {
    dataset: {role: Path(dataset_path(f'hwangbj_{dataset}_{role}'))
              for role in ['counts', 'metadata']}
    for dataset in ['liver_nuclei', 'pbmc']
}
rscript = os.environ.get('SLC7A5_RSCRIPT') or shutil.which('Rscript')
if not rscript:
    raise RuntimeError('Rscript를 PATH에 추가하거나 SLC7A5_RSCRIPT에 실행 파일을 지정하세요.')
DATASET = 'liver_nuclei'  # 첫 섹션은 간 핵; 아래 PBMC 섹션도 자동 실행
DONORS = ['MASL1', 'MASL2', 'MASL3', 'MASL4']
SEED = 42
BATCH_KEY = 'donor'
HARMONY_THETA = 2.0
HARMONY_MAX_ITER = 20
import harmonypy as hm
if hm.__version__ != '0.0.10':
    raise RuntimeError('이 분석은 harmonypy 0.0.10 기준입니다. requirements-python.txt를 사용하세요.')
print('Harmony backend:', hm.__version__, '|', hm.__file__)
MIN_GENES = 200
MIN_CELLS_PER_GENE = 3
MAX_MT_PCT = 10.0        # 입력 데이터 QC를 확인한 뒤 조정; upper-count 제거는 자동 적용하지 않음
ALL_RESOLUTION = 0.6
MYELOID_RESOLUTION = 0.5
# Cluster ID는 간 핵/PBMC에서 다르므로 dataset별로 설정합니다.
CLUSTER_OVERRIDES = {
    'liver_nuclei': {'myeloid': None, 'keep_reclusters': None, 'monocytes': None},
    'pbmc': {'myeloid': None, 'keep_reclusters': None, 'monocytes': None},
}
MYELOID_CLUSTERS = CLUSTER_OVERRIDES[DATASET]['myeloid']
MYELOID_KEEP_RECLUSTERS = CLUSTER_OVERRIDES[DATASET]['keep_reclusters']
MONOCYTE_CLUSTERS = CLUSTER_OVERRIDES[DATASET]['monocytes']
SLC_POSITIVE_QUANTILE = 0.75
MIN_POSITIVE_FOR_CUTOFF = 20
MIN_CELLS_PSEUDOBULK = 20
ANALYSIS = PROJECT / '02_Analysis' / 'HwangBJ' / DATASET / 'harmony'
FIGURES = PROJECT / '05_Figure' / 'HwangBJ' / DATASET / 'harmony'
for path in [ANALYSIS, FIGURES]:
    path.mkdir(parents=True, exist_ok=True)
sc.settings.verbosity = 2
sc.settings.n_jobs = 4
sc.set_figure_params(dpi=100, dpi_save=200, frameon=False)
print('Python:', platform.python_version(), '| Scanpy:', sc.__version__, '| AnnData:', ad.__version__)
print('Dataset:', DATASET, '| donors:', DONORS)

inventory = pd.DataFrame([
    {'dataset': dataset, 'role': role, 'file': f'{dataset}_{suffix}.rds',
     'exists': RDS_INPUTS[dataset][role].is_file()}
    for dataset in ['liver_nuclei', 'pbmc']
    for role, suffix in [('counts', 'counts_matrix'), ('metadata', 'metadata')]
])
display(inventory)
assert inventory.exists.all(), '두 dataset의 counts/metadata RDS 4개를 확인하세요.'


## 1. RDS를 sparse AnnData로 읽기

R의 dgCMatrix를 sparse binary cache로 변환하여 dense matrix 생성 없이 읽습니다. 원본 파일은 수정하지 않습니다.
Cache는 원본 경로·크기·수정 시각과 선택 donor가 일치할 때 재사용합니다.

In [ ]:
counts_path = RDS_INPUTS[DATASET]['counts']
metadata_path = RDS_INPUTS[DATASET]['metadata']
assert DATASET in ['liver_nuclei', 'pbmc']
assert counts_path.exists() and metadata_path.exists(), 'counts/metadata RDS 입력을 확인하세요.'
export_script = HWANG_SCRIPTS / 'HwangBJ_export_counts.R'
cache = ANALYSIS.parent / 'sparse_cache'  # 기존 RDS cache 재사용
cache.mkdir(exist_ok=True)
manifest = {
    'sources': [{'path': str(p), 'size': p.stat().st_size, 'mtime_ns': p.stat().st_mtime_ns}
                for p in [counts_path, metadata_path]],
    'donors': DONORS,
    'converter_sha256': hashlib.sha256(export_script.read_bytes()).hexdigest(),
}
manifest_path = cache / 'manifest.json'
cache_files = ['indices.i32', 'indptr.i32', 'values.f64', 'shape.txt', 'genes.txt', 'metadata.csv']
if not (manifest_path.exists() and json.loads(manifest_path.read_text()) == manifest
        and all((cache / name).exists() for name in cache_files)):
    result = subprocess.run([rscript, '--vanilla', str(export_script), str(counts_path),
                             str(metadata_path), str(cache), ','.join(DONORS)],
                            capture_output=True, text=True, check=True)
    print(result.stdout)
    manifest_path.write_text(json.dumps(manifest, indent=2))

shape = tuple(np.loadtxt(cache / 'shape.txt', dtype=int).tolist())
indices = np.fromfile(cache / 'indices.i32', dtype='<i4')
indptr = np.fromfile(cache / 'indptr.i32', dtype='<i4')
values = np.fromfile(cache / 'values.f64', dtype='<f8')
assert len(indptr) == shape[1] + 1 and indptr[-1] == len(values) == len(indices)
assert np.all(np.isfinite(values)) and np.all(values >= 0) and np.all(values == np.floor(values))
counts = sp.csc_matrix((values.astype(np.float32), indices, indptr), shape=shape).T.tocsr()
metadata = pd.read_csv(cache / 'metadata.csv', index_col='cell_barcode')
genes = pd.Index((cache / 'genes.txt').read_text().splitlines(), name='gene')
assert len(metadata) == counts.shape[0] and len(genes) == counts.shape[1]
assert metadata.index.is_unique and genes.is_unique
assert set(metadata.Genotype_ID) == set(DONORS)
assert metadata.Condition.eq('MASL').all(), '이 노트북의 기본 분석은 MASL만 선택합니다.'
metadata['donor'] = pd.Categorical(metadata.Genotype_ID, categories=DONORS, ordered=True)
metadata['sample'] = metadata.Genotype_ID.astype(str)
metadata['source_dataset'] = DATASET
adata = ad.AnnData(counts, obs=metadata, var=pd.DataFrame(index=genes))
adata.layers['counts'] = adata.X.copy()
adata.uns['source_manifest'] = json.dumps(manifest)
input_counts = adata.obs.donor.value_counts().reindex(DONORS, fill_value=0)
display(adata.obs.groupby('donor', observed=False).agg(n_cells=('sample', 'size'),
        condition=('Condition', 'first')))
print(adata)
del counts, values, indices, indptr

## 2. QC, 정규화, 전체 세포 클러스터링

입력은 이미 필터링된 데이터일 수 있으므로 QC 분포와 donor별 제거 수를 먼저 확인합니다.
미토콘드리아 비율/검출 유전자 수 기준을 명시적으로 적용하며 doublet 제거를 수행했다고 간주하지 않습니다.
Batch-aware HVG와 PCA 후 `donor`를 기준으로 Harmony를 적용합니다.
`X_pca_harmony`로 neighbors/UMAP/Leiden을 계산하며 보정 전/후 donor UMAP을 함께 저장합니다.
Harmony는 PCA 공간을 보정하므로 raw UMI나 SLC7A5 발현값을 새로 만들지 않습니다.
SLC7A5는 HVG/PCA에서 제외하여 목표 유전자 하나로 cluster가 만들어지는 영향을 줄입니다.

In [ ]:
adata.var['mt'] = adata.var_names.str.upper().str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, metric in zip(axes, ['total_counts', 'n_genes_by_counts', 'pct_counts_mt']):
    sns.boxplot(data=adata.obs, x='donor', y=metric, order=DONORS, ax=ax, fliersize=1)
fig.tight_layout()
fig.savefig(FIGURES / '01_input_QC.pdf', bbox_inches='tight')
plt.show()
adata.obs['qc_keep'] = ((adata.obs.n_genes_by_counts >= MIN_GENES)
                        & (adata.obs.pct_counts_mt <= MAX_MT_PCT))
qc = pd.DataFrame({'input_n': input_counts,
     'retained_n': adata.obs.loc[adata.obs.qc_keep, 'donor'].value_counts().reindex(DONORS, fill_value=0)})
qc['removed_n'] = qc.input_n - qc.retained_n
qc.to_csv(ANALYSIS / 'donor_QC.csv')
display(qc)
assert (qc.retained_n > 0).all(), 'QC 이후 일부 donor가 사라졌습니다. QC를 검토하세요.'
adata = adata[adata.obs.qc_keep].copy()
sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)
assert 'SLC7A5' in adata.var_names, 'QC 후 SLC7A5가 없습니다. 검출 수준을 확인하세요.'
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata.copy()  # log-normalized expression, raw UMI는 layers['counts']

In [ ]:
def cluster_expression(a, key, resolution):
    sc.pp.highly_variable_genes(a, n_top_genes=min(2000, a.n_vars - 1),
                               flavor='seurat', batch_key='donor')
    a.var.loc[a.var_names.isin(['SLC7A5']), 'highly_variable'] = False
    hvg = a[:, a.var.highly_variable].copy()
    sc.pp.scale(hvg, max_value=10)
    n_pcs = min(30, hvg.n_vars - 1, hvg.n_obs - 1)
    sc.tl.pca(hvg, n_comps=n_pcs, random_state=SEED, svd_solver='arpack')
    a.obsm['X_pca'] = hvg.obsm['X_pca'].copy()
    a.uns['pca'] = hvg.uns['pca'].copy()
    # 보정 전 donor UMAP을 진단용으로 유지합니다.
    sc.pp.neighbors(a, n_neighbors=min(15, a.n_obs - 1), n_pcs=n_pcs,
                    use_rep='X_pca', key_added='uncorrected_neighbors', random_state=SEED)
    sc.tl.umap(a, neighbors_key='uncorrected_neighbors', random_state=SEED)
    a.obsm['X_umap_uncorrected'] = a.obsm['X_umap'].copy()
    assert a.obs[BATCH_KEY].nunique() >= 2, 'Harmony에는 여러 donor가 필요합니다.'
    sc.external.pp.harmony_integrate(a, key=BATCH_KEY, basis='X_pca',
        adjusted_basis='X_pca_harmony', theta=HARMONY_THETA,
        max_iter_harmony=HARMONY_MAX_ITER, random_state=SEED,
        nclust=max(2, min(100, round(a.n_obs / 30))))
    assert a.obsm['X_pca_harmony'].shape == a.obsm['X_pca'].shape
    assert np.isfinite(a.obsm['X_pca_harmony']).all()
    a.uns['harmony_configuration'] = {
        'batch_key': BATCH_KEY, 'theta': HARMONY_THETA, 'max_iter_harmony': HARMONY_MAX_ITER,
        'seed': SEED, 'harmonypy_version': hm.__version__, 'neighbors_representation': 'X_pca_harmony'}
    sc.pp.neighbors(a, n_neighbors=min(15, a.n_obs - 1), n_pcs=n_pcs,
                    use_rep='X_pca_harmony', random_state=SEED)
    sc.tl.umap(a, random_state=SEED)
    sc.tl.leiden(a, resolution=resolution, key_added=key, random_state=SEED,
                 flavor='igraph', n_iterations=2, directed=False)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sc.pl.embedding(a, basis='umap_uncorrected', color=BATCH_KEY, ax=axes[0],
                    title='Before Harmony', show=False, legend_loc=None)
    sc.pl.umap(a, color=BATCH_KEY, ax=axes[1], title='After Harmony', show=False)
    fig.tight_layout()
    save_current(f'{key}_donor_before_after_Harmony')
    return a

def expr(a, gene):
    if gene not in a.var_names:
        return np.zeros(a.n_obs)
    x = a[:, gene].X
    return x.toarray().ravel() if sp.issparse(x) else np.asarray(x).ravel()

def save_current(name):
    plt.gcf().savefig(FIGURES / f'{name}.pdf', bbox_inches='tight')
    plt.gcf().savefig(FIGURES / f'{name}.png', dpi=200, bbox_inches='tight')
    plt.show()

def marker_dotplot(a, panels, groupby, filename):
    if a.n_obs == 0:
        # 후보가 없어 새 plot을 만들지 않을 때 이전 후보의 그림이 남지 않도록 보관합니다.
        previous = FIGURES / f'{filename}.pdf'
        if previous.exists():
            archive = ANALYSIS / 'previous_outputs'
            archive.mkdir(exist_ok=True)
            previous.replace(archive / f'{filename}_previous.pdf')
        print(f'{filename}: 선택된 후보가 없어 dotplot을 생략합니다.')
        return
    present = {k: [g for g in v if g in a.var_names] for k, v in panels.items()}
    present = {k: v for k, v in present.items() if v}
    dp = sc.pl.dotplot(a, present, groupby=groupby, use_raw=False,
                      standard_scale='var', return_fig=True)
    dp.savefig(str(FIGURES / f'{filename}.pdf'))
    dp.show()

def score_panels(a, panels, prefix):
    keys = []
    for name, genes in panels.items():
        genes = [g for g in genes if g in a.var_names]
        if len(genes) >= 2:
            key = prefix + name
            sc.tl.score_genes(a, genes, score_name=key, use_raw=False, random_state=SEED)
            keys.append(key)
    return keys

cluster_expression(adata, 'all_cluster', ALL_RESOLUTION)
sc.pl.umap(adata, color=['donor', 'all_cluster'], ncols=2, show=False)
save_current('02_all_UMAP_donor_cluster')

## 3. Marker로 myeloid 후보 선택

여러 lineage signature를 동일한 방식으로 점수화합니다. 자동 선택은 cluster의 평균 myeloid score가
다른 lineage보다 높고 양수인 경우입니다. NK와 공유하는 TYROBP/FCER1G만으로 myeloid를 판단하지 않도록
SPI1/CSF1R/LYZ/LST1 및 경쟁 lineage marker를 함께 검토합니다.
`MYELOID_CLUSTERS`를 수동 지정하면 그 선택을 우선합니다. 점수는 세포형 판정의 통계 검정이 아닙니다.

In [ ]:
LINEAGE = {
    'Myeloid': ['SPI1', 'CSF1R', 'LYZ', 'LST1', 'AIF1', 'CTSS', 'FCER1G', 'TYROBP'],
    'T_cell': ['CD3D', 'CD3E', 'TRAC', 'CD247'],
    'NK': ['NKG7', 'GNLY', 'PRF1', 'KLRD1', 'CTSW'],
    'B_cell': ['MS4A1', 'CD79A', 'CD79B', 'CD37'],
    'Hepatocyte': ['ALB', 'TTR', 'APOA1', 'ASGR1', 'CPS1'],
    'Cholangiocyte': ['KRT19', 'EPCAM', 'KRT7', 'SOX9'],
    'Endothelial': ['PECAM1', 'VWF', 'EMCN', 'KDR', 'CLEC4G'],
    'Stellate': ['COL1A1', 'COL1A2', 'DCN', 'LUM', 'PDGFRB'],
}
lineage_keys = score_panels(adata, LINEAGE, 'lineage_')
lineage_summary = adata.obs.groupby('all_cluster', observed=True)[lineage_keys].mean()
lineage_summary['dominant_signature'] = lineage_summary[lineage_keys].idxmax(axis=1).str.replace('lineage_', '')
lineage_summary['n_cells'] = adata.obs.all_cluster.value_counts()
lineage_summary.to_csv(ANALYSIS / 'all_cluster_lineage_scores.csv')
display(lineage_summary)
marker_dotplot(adata, LINEAGE, 'all_cluster', '03_all_lineage_markers')
automatic_myeloid = lineage_summary.index[(lineage_summary.dominant_signature == 'Myeloid')
                                          & (lineage_summary.lineage_Myeloid > 0)].tolist()
selected_myeloid = automatic_myeloid if MYELOID_CLUSTERS is None else list(map(str, MYELOID_CLUSTERS))
assert set(selected_myeloid).issubset(set(adata.obs.all_cluster)), 'Myeloid cluster ID 확인'
adata.obs['myeloid_candidate'] = adata.obs.all_cluster.isin(selected_myeloid)
adata.obs['lineage_candidate'] = adata.obs.all_cluster.map(lineage_summary.dominant_signature).astype(str)
print('Selected myeloid candidate clusters:', selected_myeloid)
display(pd.crosstab(adata.obs.donor, adata.obs.myeloid_candidate).reindex(DONORS, fill_value=0))
donor_mix = pd.crosstab(adata.obs.all_cluster, adata.obs.donor).reindex(columns=DONORS, fill_value=0)
donor_mix.to_csv(ANALYSIS / 'all_cluster_donor_counts.csv')
sc.pl.umap(adata, color=['lineage_candidate', 'myeloid_candidate', 'SLC7A5'], ncols=3, show=False)
save_current('04_all_lineage_SLC7A5')
assert adata.obs.myeloid_candidate.sum() >= 30, 'Myeloid 후보가 너무 적습니다. 전체 marker와 cluster 선택을 확인하세요.'
myeloid = adata[adata.obs.myeloid_candidate].copy()

## 4. Myeloid 재클러스터링 및 monocyte 후보

Figure 2 marker 구성을 참조하되 `Special_monocyte`를 자동 주석의 선택지에 넣지 않습니다.
Classical/non-classical monocyte와 Kupffer/LAM/DC를 먼저 구분합니다.
Monocyte 후보는 monocyte signature가 우세한 cluster로 정의하고, 이후 SLC7A5 발현 분할을 별도로 수행합니다.
SLC7A5를 제외한 PPIF/ANPEP/ITGAX/VCAN signature는 기존 특성과의 일치도를 확인하는 보조 지표입니다.
재클러스터링 후 canonical myeloid marker 6개 중 3개 이상이 cluster 핵의 10% 이상에서 검출되는지
검토하여 부족한 cluster를 제외합니다. 이는 명시적 탐색적 QC 기준이며 수동 선택으로 바꿀 수 있습니다.
Classical 후보에서 CD14/FCN1/VCAN 모두 10% 미만, 또는 non-classical 후보에서 FCGR3A가 10% 미만
검출되면 `Monocyte_DC_ambiguous`로 표시합니다. 이 기준은 확정 주석 기준이 아니라 검토 필요 표시입니다.
Myeloid에서도 HVG/PCA를 새로 계산하고 donor Harmony를 다시 적용합니다. 전체 세포에서 얻은 보정 좌표를 그대로 재사용하지 않습니다.
재클러스터링 후에도 경쟁 lineage score가 myeloid보다 우세한 cluster는 자동 선택에서 제외합니다.
모든 subtype score가 0 이하인 cluster는 Unresolved로 남기고, 단순히 가장 큰 점수가 monocyte라는 이유로 monocyte에 포함하지 않습니다.

In [ ]:
MYELOID_MARKERS = {
    'Classical_monocyte': ['S100A9', 'S100A8', 'S100A6', 'LYZ', 'FCN1', 'CD14', 'VCAN'],
    'Nonclassical_monocyte': ['IFITM2', 'IFITM3', 'CD52', 'LST1', 'FCGR3A'],
    'Kupffer_cell': ['CD5L', 'CD163', 'SLC40A1', 'MARCO', 'VCAM1'],
    'LAM': ['TREM2', 'LGALS3', 'C1QC', 'APOC1', 'GPNMB'],
    'Macrophage_C1Q': ['C1QA', 'C1QB', 'C1QC', 'AIF1', 'APOE'],
    'cDC1': ['SNX3', 'DNASE1L3', 'CLEC9A', 'IRF8', 'IDO1'],
    'cDC2': ['HLA-DQB1', 'HLA-DPB1', 'HLA-DQA1', 'CD1C', 'FCER1A'],
    'Migratory_DC': ['GBP1', 'IL4I1', 'CXCL10', 'LAMP3', 'CD83'],
}
cluster_expression(myeloid, 'myeloid_cluster', MYELOID_RESOLUTION)
canonical = [g for g in ['SPI1', 'CSF1R', 'LST1', 'CTSS', 'FCER1G', 'TYROBP'] if g in myeloid.var_names]
canonical_detection = pd.DataFrame({g: expr(myeloid, g) > 0 for g in canonical}, index=myeloid.obs_names)
canonical_detection['cluster'] = myeloid.obs.myeloid_cluster.to_numpy()
canonical_audit = canonical_detection.groupby('cluster', observed=True).mean()
canonical_audit['n_markers_detected_in_10pct'] = (canonical_audit[canonical] >= 0.10).sum(axis=1)
canonical_audit['n_cells'] = myeloid.obs.myeloid_cluster.value_counts()
# Myeloid 전체 cluster에 섞인 간세포 등은 재클러스터링 후 경쟁 lineage도 확인합니다.
recluster_lineage = myeloid.obs.groupby('myeloid_cluster', observed=True)[lineage_keys].mean()
canonical_audit['dominant_lineage'] = recluster_lineage.idxmax(axis=1).str.replace('lineage_', '')
canonical_audit['mean_lineage_Myeloid'] = recluster_lineage['lineage_Myeloid']
auto_keep = canonical_audit.index[(canonical_audit.n_markers_detected_in_10pct >= 3)
    & (canonical_audit.dominant_lineage == 'Myeloid')
    & (canonical_audit.mean_lineage_Myeloid > 0)].tolist()
keep_reclusters = auto_keep if MYELOID_KEEP_RECLUSTERS is None else list(map(str, MYELOID_KEEP_RECLUSTERS))
assert set(keep_reclusters).issubset(set(myeloid.obs.myeloid_cluster))
canonical_audit['retained'] = canonical_audit.index.isin(keep_reclusters)
canonical_audit.to_csv(ANALYSIS / 'myeloid_recluster_marker_QC.csv')
display(canonical_audit)
excluded_ids = myeloid.obs_names[~myeloid.obs.myeloid_cluster.isin(keep_reclusters)]
adata.obs.loc[excluded_ids, 'myeloid_candidate'] = False
adata.obs['lineage_candidate'] = adata.obs.lineage_candidate.astype(str)
adata.obs.loc[excluded_ids, 'lineage_candidate'] = 'Excluded_from_myeloid_marker_review'
print('Excluded after myeloid marker review:', len(excluded_ids))
myeloid = myeloid[myeloid.obs.myeloid_cluster.isin(keep_reclusters)].copy()
assert myeloid.n_obs >= 30, 'Marker-supported myeloid 후보가 부족합니다. QC를 검토하세요.'
subtype_keys = score_panels(myeloid, MYELOID_MARKERS, 'subtype_')
subtype_summary = myeloid.obs.groupby('myeloid_cluster', observed=True)[subtype_keys].mean()
subtype_summary['dominant_signature'] = subtype_summary[subtype_keys].idxmax(axis=1).str.replace('subtype_', '')
subtype_summary['n_cells'] = myeloid.obs.myeloid_cluster.value_counts()
subtype_summary['annotation_candidate'] = subtype_summary.dominant_signature
weak_all = subtype_summary[subtype_keys].max(axis=1) <= 0
subtype_summary.loc[weak_all, 'annotation_candidate'] = 'Unresolved_myeloid'
mono_check = ['CD14', 'FCN1', 'VCAN', 'FCGR3A']
mono_detection = pd.DataFrame({g: expr(myeloid, g) > 0 for g in mono_check}, index=myeloid.obs_names)
mono_detection['cluster'] = myeloid.obs.myeloid_cluster.to_numpy()
mono_detection = mono_detection.groupby('cluster', observed=True).mean()
mono_detection.to_csv(ANALYSIS / 'myeloid_cluster_monocyte_marker_detection.csv')
is_mono_signature = subtype_summary.dominant_signature.isin(['Classical_monocyte', 'Nonclassical_monocyte'])
weak_classical = (subtype_summary.dominant_signature == 'Classical_monocyte') & (
    mono_detection[['CD14', 'FCN1', 'VCAN']].max(axis=1) < 0.10)
weak_nonclassical = (subtype_summary.dominant_signature == 'Nonclassical_monocyte') & (
    mono_detection.FCGR3A < 0.10)
weak_mono = (weak_classical | weak_nonclassical) & ~weak_all
subtype_summary.loc[weak_mono, 'annotation_candidate'] = 'Monocyte_DC_ambiguous'
subtype_summary['monocyte_marker_support_weak'] = weak_mono
subtype_summary.to_csv(ANALYSIS / 'myeloid_cluster_signature_scores.csv')
display(subtype_summary)
marker_dotplot(myeloid, {**MYELOID_MARKERS,
    'SLC7A5_high_reference': ['PPIF', 'ANPEP', 'SLC7A5', 'ITGAX', 'VCAN']},
    'myeloid_cluster', '05_myeloid_marker_dotplot')
auto_mono = subtype_summary.index[subtype_summary.dominant_signature.isin(
                                  ['Classical_monocyte', 'Nonclassical_monocyte']) & ~weak_all].tolist()
selected_mono = auto_mono if MONOCYTE_CLUSTERS is None else list(map(str, MONOCYTE_CLUSTERS))
assert set(selected_mono).issubset(set(myeloid.obs.myeloid_cluster)), 'Monocyte cluster ID 확인'
myeloid.obs['subtype_candidate'] = myeloid.obs.myeloid_cluster.map(subtype_summary.annotation_candidate).astype(str)
myeloid.obs['monocyte_candidate'] = myeloid.obs.myeloid_cluster.isin(selected_mono)
sc.tl.score_genes(myeloid, [g for g in ['PPIF', 'ANPEP', 'ITGAX', 'VCAN'] if g in myeloid.var_names],
                  score_name='reference_signature_without_SLC7A5', use_raw=False, random_state=SEED)
myeloid.obs['SLC7A5_log1p'] = expr(myeloid, 'SLC7A5')
sc.pl.umap(myeloid, color=['donor', 'myeloid_cluster', 'subtype_candidate', 'SLC7A5',
                           'reference_signature_without_SLC7A5', 'monocyte_candidate'], ncols=3, show=False)
save_current('06_myeloid_UMAP')
print('Selected monocyte candidate clusters:', selected_mono)
if subtype_summary.loc[selected_mono, 'monocyte_marker_support_weak'].any():
    print('주의: 선택된 후보에 monocyte/DC 구분이 불명확한 cluster가 있습니다. 확정 monocyte 주석으로 사용하지 마세요.')
# Cluster marker ranking is descriptive: no cell-level P values are exported or used for inference.
sc.tl.rank_genes_groups(myeloid, groupby='myeloid_cluster', method='t-test',
                       use_raw=False, n_genes=30, key_added='cluster_marker_ranking')
ranked = sc.get.rank_genes_groups_df(myeloid, group=None, key='cluster_marker_ranking')
ranked[['group', 'names', 'logfoldchanges']].to_csv(ANALYSIS / 'myeloid_top30_markers_descriptive.csv', index=False)
mix = pd.crosstab(myeloid.obs.myeloid_cluster, myeloid.obs.donor).reindex(columns=DONORS, fill_value=0)
mix.to_csv(ANALYSIS / 'myeloid_cluster_donor_counts.csv')
display(mix)
if myeloid.obs.monocyte_candidate.sum() < 20:
    print('Monocyte 후보가 20개 미만입니다. marker와 MONOCYTE_CLUSTERS를 검토하세요. 부족한 후보를 임의로 채우지 않습니다.')
monocytes = myeloid[myeloid.obs.monocyte_candidate].copy()

## 5. SLC7A5-high 발현 그룹과 기존 phenotype 비교

**주 분석의 표현형 정의:** 선택된 전체 monocyte 후보에서 SLC7A5가 검출된 핵의 log-normalized 발현 75백분위 이상.
네 donor에 하나의 pooled cutoff를 적용합니다. 양성 핵이 20개 미만이면 안정적인 high cutoff를 만들지 않고
`detected_only_insufficient_for_high`로 별도 표시합니다. 0 발현은 high에 포함하지 않습니다.
이는 탐색적 발현 그룹이며 기존 Figure의 `Special_monocyte` 주석과 동일한 정의가 아닙니다.

**보조 gate:** 기존 핵 분석에서 사용한 CD14-high / FCGR3A-low / SLC7A5-high 조합을 별도 표시합니다.
이 데이터에는 Healthy가 없으므로 Healthy 기준 cutoff는 만들 수 없습니다. CD14 양성값 75백분위와 전체
FCGR3A 25백분위를 사용한 탐색적 gate이며 주 monocyte 정의를 대체하지 않습니다.
High cutoff는 간 핵과 PBMC에서 각각 계산하며 검체 간 동일한 high 집단으로 해석하지 않습니다.

In [ ]:
slc = expr(monocytes, 'SLC7A5')
positive = slc[slc > 0]
high_available = len(positive) >= MIN_POSITIVE_FOR_CUTOFF
slc_cutoff = float(np.quantile(positive, SLC_POSITIVE_QUANTILE)) if high_available else None
monocytes.obs['SLC7A5_detected'] = slc > 0
monocytes.obs['SLC7A5_high_candidate'] = (slc >= slc_cutoff) & (slc > 0) if high_available else False
monocytes.obs['SLC_expression_group'] = np.where(monocytes.obs.SLC7A5_high_candidate,
    'SLC7A5_high_candidate', 'Other_monocytes') if high_available else np.where(slc > 0,
    'detected_only_insufficient_for_high', 'Undetected')
cd14, fcgr3a = expr(monocytes, 'CD14'), expr(monocytes, 'FCGR3A')
cd14_positive = cd14[cd14 > 0]
gate_available = high_available and len(cd14_positive) >= MIN_POSITIVE_FOR_CUTOFF
cd14_cutoff = float(np.quantile(cd14_positive, 0.75)) if gate_available else None
fcgr3a_cutoff = float(np.quantile(fcgr3a, 0.25)) if gate_available else None
monocytes.obs['triple_gate_candidate'] = (
    monocytes.obs.SLC7A5_high_candidate & (cd14 >= cd14_cutoff) & (fcgr3a <= fcgr3a_cutoff)
) if gate_available else False
definition = {'group': 'exploratory expression-defined candidate, not confirmed Special_monocyte',
    'monocyte_candidate_n': monocytes.n_obs, 'monocyte_definition_available': bool(monocytes.n_obs),
    'quantile_among_positive_monocytes': SLC_POSITIVE_QUANTILE,
    'minimum_positive_n': MIN_POSITIVE_FOR_CUTOFF, 'positive_n': len(positive),
    'high_definition_available': bool(high_available), 'SLC7A5_log1p_cutoff': slc_cutoff,
    'triple_gate_available': bool(gate_available), 'CD14_positive_q75': cd14_cutoff,
    'FCGR3A_all_q25': fcgr3a_cutoff,
    'myeloid_clusters': selected_myeloid, 'monocyte_clusters': selected_mono}
definition['retained_myeloid_reclusters'] = keep_reclusters
definition['weak_monocyte_marker_clusters'] = subtype_summary.index[
    subtype_summary.monocyte_marker_support_weak].tolist()
(ANALYSIS / 'SLC7A5_candidate_definition.json').write_text(json.dumps(definition, indent=2))
print(json.dumps(definition, indent=2))
monocytes.obs.to_csv(ANALYSIS / 'monocyte_cell_metadata.csv')
for column in ['SLC7A5_detected', 'SLC7A5_high_candidate', 'triple_gate_candidate']:
    myeloid.obs[column] = False
    myeloid.obs.loc[monocytes.obs_names, column] = monocytes.obs[column].to_numpy()
sc.pl.umap(myeloid, color=['SLC7A5', 'SLC7A5_high_candidate', 'triple_gate_candidate'], ncols=3, show=False)
save_current('07_SLC7A5_candidates_UMAP')
marker_dotplot(monocytes, {'Monocyte': ['CD14', 'FCGR3A', 'FCN1', 'S100A8', 'S100A9'],
    'Reference': ['PPIF', 'ANPEP', 'SLC7A5', 'ITGAX', 'VCAN'],
    'Macrophage_DC_check': ['C1QA', 'MARCO', 'TREM2', 'CD1C', 'CLEC9A']},
    'SLC_expression_group', '08_SLC7A5_candidate_marker_check')

## 6. Donor별 세포 수·비율 및 signature

각 donor의 high 후보 수를 해당 donor의 **전체 monocyte 후보 수**로 나눈 비율을 주 지표로 표시합니다.
전체 QC 통과 핵과 myeloid 후보를 분모로 한 비율도 함께 저장합니다. high 후보가 없는 donor도 0으로 포함합니다.
그림의 `n`은 donor별 포착된 핵 수이며 생물학적 반복 수는 donor 4명입니다.
비율은 이 데이터에서 회수된 핵의 조성으로 해석합니다.

In [ ]:
def donor_n(a, mask=None):
    obs = a.obs if mask is None else a.obs.loc[mask]
    return obs.donor.value_counts().reindex(DONORS, fill_value=0).astype(int)

fractions = pd.DataFrame({'total_QC_n': donor_n(adata), 'myeloid_n': donor_n(myeloid),
    'monocyte_n': donor_n(monocytes), 'SLC7A5_detected_n': donor_n(monocytes, monocytes.obs.SLC7A5_detected),
    'high_n': donor_n(monocytes, monocytes.obs.SLC7A5_high_candidate),
    'triple_gate_n': donor_n(monocytes, monocytes.obs.triple_gate_candidate)})
fractions['detected_pct_of_monocytes'] = 100 * fractions.SLC7A5_detected_n / fractions.monocyte_n.replace(0, np.nan)
for denom in ['monocyte', 'myeloid', 'total_QC']:
    fractions[f'high_pct_of_{denom}'] = (100 * fractions.high_n / fractions[f'{denom}_n'].replace(0, np.nan)
                                        if high_available else np.nan)
fractions['triple_gate_pct_of_monocytes'] = (100 * fractions.triple_gate_n / fractions.monocyte_n.replace(0, np.nan)
                                            if gate_available else np.nan)
fractions['high_definition_available'] = high_available
fractions['triple_gate_available'] = gate_available
if not high_available:
    fractions['high_n'] = np.nan  # 정의 불가를 0개의 high 집단으로 해석하지 않도록 표시
if not gate_available:
    fractions['triple_gate_n'] = np.nan
fractions.index.name = 'donor' 
fractions.to_csv(ANALYSIS / 'donor_SLC7A5_monocyte_frequencies.csv')
display(fractions)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
ycolumn = 'high_pct_of_monocyte' if high_available else 'detected_pct_of_monocytes'
axes[0].scatter(np.arange(len(DONORS)), fractions[ycolumn], s=65, color='#bd4337')
axes[0].set_xticks(np.arange(len(DONORS)), DONORS)
axes[0].set_ylabel('Monocyte candidates (%)', fontsize=10)
axes[0].set_title('SLC7A5-high candidates' if high_available else 'SLC7A5 detected', fontsize=11)
axes[0].set_ylim(bottom=0)
if monocytes.n_obs == 0:
    axes[0].text(0.5, 0.5, 'No marker-supported\nmonocyte candidates',
                 ha='center', va='center', transform=axes[0].transAxes, fontsize=11)
    axes[0].set_title('Monocyte definition unavailable', fontsize=11)
    axes[0].set_ylabel('')
    axes[0].set_yticks([])
for i, donor in enumerate(DONORS):
    if pd.notna(fractions.loc[donor, ycolumn]):
        numerator = fractions.loc[donor, 'high_n' if high_available else 'SLC7A5_detected_n']
        axes[0].annotate(f'{numerator}/{fractions.loc[donor, "monocyte_n"]}',
                         (i, fractions.loc[donor, ycolumn]), xytext=(0, 8), textcoords='offset points', ha='center', fontsize=8)
composition = pd.crosstab(myeloid.obs.donor, myeloid.obs.subtype_candidate).reindex(DONORS, fill_value=0)
composition.div(composition.sum(axis=1), axis=0).mul(100).plot.bar(stacked=True, ax=axes[1], width=0.8)
axes[1].set_ylabel('Myeloid candidates (%)', fontsize=10)
axes[1].set_title('Candidate subtype composition', fontsize=11)
for axis in axes:
    axis.tick_params(labelsize=10)
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
axes[1].tick_params(axis='x', rotation=0)
fig.tight_layout()
save_current('09_donor_SLC7A5_frequencies_and_myeloid_composition')
composition.to_csv(ANALYSIS / 'donor_myeloid_subtype_counts.csv')

# Descriptive donor-level expression/signature summaries; no cell-level hypothesis testing.
summary = monocytes.obs.groupby(['donor', 'SLC_expression_group'], observed=True).agg(
    n_cells=('sample', 'size'), mean_SLC7A5_log1p=('SLC7A5_log1p', 'mean'),
    mean_reference_signature=('reference_signature_without_SLC7A5', 'mean')).reset_index()
summary.to_csv(ANALYSIS / 'donor_candidate_expression_summary.csv', index=False)
display(summary)
if high_available:
    fig, ax = plt.subplots(figsize=(5.5, 4))
    for donor, group in summary.groupby('donor', observed=True):
        group = group.set_index('SLC_expression_group').reindex(['Other_monocytes', 'SLC7A5_high_candidate'])
        ax.plot([0, 1], group.mean_reference_signature, marker='o', label=str(donor))
    ax.set_xticks([0, 1], ['Other monocytes', 'SLC7A5-high candidate'])
    ax.set_ylabel('Mean reference signature score (PPIF / ANPEP / ITGAX / VCAN)')
    ax.legend(title='Donor', bbox_to_anchor=(1.02, 1))
    fig.tight_layout()
    save_current('10_donor_reference_signature')

## 7. 저장 및 후속 donor-level 분석 준비

전체/myeloid/monocyte AnnData와 raw-count pseudobulk를 저장합니다. Pseudobulk는 donor × high/other의 UMI 합입니다.
0-cell 조합은 표에 포함하되 DE 분석용 matrix에서 제외하고, `eligible_for_DE`에 최소 핵 수 조건을 기록합니다.
후속 high-vs-other DE에는 donor를 paired design에 포함하고, 충분한 group별 핵 수가 있는 donor만 사용해야 합니다.
SLC7A5 자체는 그룹 정의에 사용했으므로 독립적인 발견 DEG로 해석하지 않습니다.
MASL 대 MASH 또는 Healthy 비교는 해당 donor를 추가한 별도 분석이 필요합니다.

In [ ]:
adata.uns['analysis_configuration'] = json.dumps({'dataset': DATASET, 'donors': DONORS,
    'min_genes': MIN_GENES, 'max_mt_pct': MAX_MT_PCT, 'seed': SEED,
    'all_resolution': ALL_RESOLUTION, 'myeloid_resolution': MYELOID_RESOLUTION,
    'batch_correction': 'Harmony', 'batch_key': BATCH_KEY,
    'harmony_theta': HARMONY_THETA, 'harmonypy_version': hm.__version__, 'annotation_status': 'marker-based provisional'})
coverage = {name: {'present': [g for g in genes if g in adata.var_names],
                   'missing_after_gene_filter': [g for g in genes if g not in adata.var_names]}
            for name, genes in {**LINEAGE, **MYELOID_MARKERS}.items()}
(ANALYSIS / 'marker_panel_coverage.json').write_text(json.dumps(coverage, indent=2))
myeloid.uns['candidate_definition'] = json.dumps(definition)
monocytes.uns['candidate_definition'] = json.dumps(definition)
for obj, filename in [(adata, 'MASL1_4_all_QC.h5ad'), (myeloid, 'MASL1_4_myeloid_candidates.h5ad'),
                       (monocytes, 'MASL1_4_monocyte_candidates.h5ad')]:
    # Marker ranking P values are not retained in delivered analysis objects.
    obj.uns.pop('cluster_marker_ranking', None)
    obj.write_h5ad(ANALYSIS / filename, compression='gzip')

if high_available:
    groups = ['Other_monocytes', 'SLC7A5_high_candidate']
    rows, meta_rows = [], []
    for donor in DONORS:
        for group in groups:
            mask = (monocytes.obs.donor == donor) & (monocytes.obs.SLC_expression_group == group)
            n = int(mask.sum())
            meta_rows.append({'pseudobulk_id': f'{donor}__{group}', 'donor': donor, 'group': group,
                             'n_cells': n, 'eligible_for_DE': n >= MIN_CELLS_PSEUDOBULK,
                             'included_in_matrix': n > 0})
            if n:
                rows.append(sp.csr_matrix(np.asarray(monocytes.layers['counts'][mask.to_numpy()].sum(axis=0), dtype=np.int64)))
    pb_metadata = pd.DataFrame(meta_rows)
    pb_counts = sp.vstack(rows, format='csr')
    sp.save_npz(ANALYSIS / 'monocyte_donor_group_pseudobulk_counts.npz', pb_counts)
    pd.Series(monocytes.var_names, name='gene').to_csv(ANALYSIS / 'pseudobulk_genes.csv', index=False)
    pb_metadata.to_csv(ANALYSIS / 'pseudobulk_metadata_all_combinations.csv', index=False)
    pb_metadata.loc[pb_metadata.included_in_matrix].to_csv(ANALYSIS / 'pseudobulk_matrix_rows.csv', index=False)
    display(pb_metadata)
    assert int(pb_counts.sum()) == int(monocytes.layers['counts'].sum(dtype=np.float64)), 'Pseudobulk count conservation failed'

report = {'dataset': DATASET, 'batch_correction': 'Harmony', 'batch_key': BATCH_KEY,
    'donors': DONORS, 'input_n': int(input_counts.sum()),
    'QC_n': adata.n_obs, 'myeloid_candidate_n': myeloid.n_obs, 'monocyte_candidate_n': monocytes.n_obs,
    'SLC7A5_detected_n': int(monocytes.obs.SLC7A5_detected.sum()) if monocytes.n_obs else None,
    'monocyte_definition_status': 'marker_based_candidate' if monocytes.n_obs else 'no_automatic_marker_supported_candidates',
    'SLC7A5_high_candidate_n': int(monocytes.obs.SLC7A5_high_candidate.sum()) if high_available else None,
    'high_definition_available': bool(high_available), 'annotation_status': 'provisional; marker review required',
    'weak_monocyte_marker_clusters': definition['weak_monocyte_marker_clusters'],
    'statistical_testing': 'none; four MASL donors, descriptive composition only'}
(ANALYSIS / 'analysis_summary.json').write_text(json.dumps(report, indent=2))
display(report)
print('Analysis:', ANALYSIS)
print('Figures:', FIGURES)

# PBMC: MASL1–4 + MASH1–4 통합 donor Harmony 분석

PBMC counts/metadata RDS의 **MASL 4명 + MASH 4명**을 모두 포함합니다.
간 핵과는 별도로, PBMC 안에서 donor Harmony 및 myeloid 재클러스터링을 수행합니다.
기존 MASL-only cluster 3의 번호를 통합 코호트에 그대로 대입하지 않습니다.
질환군 차이/P 값과 무관하게 기존 scRNA-seq 참조 DEG와의 유사도로 marker-supported monocyte 집단을 선택하고,
그 주석으로 MASL 대 MASH의 donor-level 비율 차이를 검정합니다.
원본 RDS와 이전 MASL-only 분석 결과는 보존하며 새 결과는 `harmony_MASL_MASH/`에 저장합니다.


In [ ]:
DATASET = 'pbmc'
DONORS = [f'MASL{i}' for i in range(1, 5)] + [f'MASH{i}' for i in range(1, 5)]
PBMC_COHORT_PREFIX = 'MASL_MASH1_4'
ANALYSIS = PROJECT / '02_Analysis' / 'HwangBJ' / DATASET / 'harmony_MASL_MASH'
FIGURES = PROJECT / '05_Figure' / 'HwangBJ' / DATASET / 'harmony_MASL_MASH'
MYELOID_CLUSTERS = CLUSTER_OVERRIDES[DATASET]['myeloid']
MYELOID_KEEP_RECLUSTERS = CLUSTER_OVERRIDES[DATASET]['keep_reclusters']
MONOCYTE_CLUSTERS = CLUSTER_OVERRIDES[DATASET]['monocytes']
for path in [ANALYSIS, FIGURES]:
    path.mkdir(parents=True, exist_ok=True)
print('Dataset:', DATASET, '| donors:', DONORS)
# 간 핵 객체를 비워 두 dataset 전체 객체가 동시에 메모리에 남는 것을 피합니다.
for name in ['adata', 'myeloid', 'monocytes']:
    globals().pop(name, None)
import gc
gc.collect()


## 1. RDS를 sparse AnnData로 읽기

R의 dgCMatrix를 sparse binary cache로 변환하여 dense matrix 생성 없이 읽습니다. 원본 파일은 수정하지 않습니다.
Cache는 원본 경로·크기·수정 시각과 선택 donor가 일치할 때 재사용합니다.

In [ ]:
counts_path = RDS_INPUTS[DATASET]['counts']
metadata_path = RDS_INPUTS[DATASET]['metadata']
assert DATASET in ['liver_nuclei', 'pbmc']
assert counts_path.exists() and metadata_path.exists(), 'counts/metadata RDS 입력을 확인하세요.'
export_script = HWANG_SCRIPTS / 'HwangBJ_export_counts.R'
cache = ANALYSIS.parent / 'sparse_cache_MASL_MASH'  # 기존 RDS cache 재사용
cache.mkdir(exist_ok=True)
manifest = {
    'sources': [{'path': str(p), 'size': p.stat().st_size, 'mtime_ns': p.stat().st_mtime_ns}
                for p in [counts_path, metadata_path]],
    'donors': DONORS,
    'converter_sha256': hashlib.sha256(export_script.read_bytes()).hexdigest(),
}
manifest_path = cache / 'manifest.json'
cache_files = ['indices.i32', 'indptr.i32', 'values.f64', 'shape.txt', 'genes.txt', 'metadata.csv']
if not (manifest_path.exists() and json.loads(manifest_path.read_text()) == manifest
        and all((cache / name).exists() for name in cache_files)):
    result = subprocess.run([rscript, '--vanilla', str(export_script), str(counts_path),
                             str(metadata_path), str(cache), ','.join(DONORS)],
                            capture_output=True, text=True, check=True)
    print(result.stdout)
    manifest_path.write_text(json.dumps(manifest, indent=2))

shape = tuple(np.loadtxt(cache / 'shape.txt', dtype=int).tolist())
indices = np.fromfile(cache / 'indices.i32', dtype='<i4')
indptr = np.fromfile(cache / 'indptr.i32', dtype='<i4')
values = np.fromfile(cache / 'values.f64', dtype='<f8')
assert len(indptr) == shape[1] + 1 and indptr[-1] == len(values) == len(indices)
assert np.all(np.isfinite(values)) and np.all(values >= 0) and np.all(values == np.floor(values))
counts = sp.csc_matrix((values.astype(np.float32), indices, indptr), shape=shape).T.tocsr()
metadata = pd.read_csv(cache / 'metadata.csv', index_col='cell_barcode')
genes = pd.Index((cache / 'genes.txt').read_text().splitlines(), name='gene')
assert len(metadata) == counts.shape[0] and len(genes) == counts.shape[1]
assert metadata.index.is_unique and genes.is_unique
assert set(metadata.Genotype_ID) == set(DONORS)
assert set(metadata.Condition) == {'MASL', 'MASH'}
assert metadata.groupby('Genotype_ID').Condition.nunique().eq(1).all()
expected_conditions = {d: 'MASL' if d.startswith('MASL') else 'MASH' for d in DONORS}
assert metadata.Condition.eq(metadata.Genotype_ID.map(expected_conditions)).all(), 'Donor/Condition 불일치'
metadata['donor'] = pd.Categorical(metadata.Genotype_ID, categories=DONORS, ordered=True)
metadata['sample'] = metadata.Genotype_ID.astype(str)
metadata['source_dataset'] = DATASET
adata = ad.AnnData(counts, obs=metadata, var=pd.DataFrame(index=genes))
adata.layers['counts'] = adata.X.copy()
adata.uns['source_manifest'] = json.dumps(manifest)
input_counts = adata.obs.donor.value_counts().reindex(DONORS, fill_value=0)
display(adata.obs.groupby('donor', observed=False).agg(n_cells=('sample', 'size'),
        condition=('Condition', 'first')))
print(adata)
del counts, values, indices, indptr

## 2. QC, 정규화, 전체 세포 클러스터링

입력은 이미 필터링된 데이터일 수 있으므로 QC 분포와 donor별 제거 수를 먼저 확인합니다.
미토콘드리아 비율/검출 유전자 수 기준을 명시적으로 적용하며 doublet 제거를 수행했다고 간주하지 않습니다.
Batch-aware HVG와 PCA 후 `donor`를 기준으로 Harmony를 적용합니다.
`X_pca_harmony`로 neighbors/UMAP/Leiden을 계산하며 보정 전/후 donor UMAP을 함께 저장합니다.
Harmony는 PCA 공간을 보정하므로 raw UMI나 SLC7A5 발현값을 새로 만들지 않습니다.
SLC7A5는 HVG/PCA에서 제외하여 목표 유전자 하나로 cluster가 만들어지는 영향을 줄입니다.

In [ ]:
adata.var['mt'] = adata.var_names.str.upper().str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, metric in zip(axes, ['total_counts', 'n_genes_by_counts', 'pct_counts_mt']):
    sns.boxplot(data=adata.obs, x='donor', y=metric, order=DONORS, ax=ax, fliersize=1)
fig.tight_layout()
fig.savefig(FIGURES / '01_input_QC.pdf', bbox_inches='tight')
plt.show()
adata.obs['qc_keep'] = ((adata.obs.n_genes_by_counts >= MIN_GENES)
                        & (adata.obs.pct_counts_mt <= MAX_MT_PCT))
qc = pd.DataFrame({'input_n': input_counts,
     'retained_n': adata.obs.loc[adata.obs.qc_keep, 'donor'].value_counts().reindex(DONORS, fill_value=0)})
qc['removed_n'] = qc.input_n - qc.retained_n
qc.to_csv(ANALYSIS / 'donor_QC.csv')
display(qc)
assert (qc.retained_n > 0).all(), 'QC 이후 일부 donor가 사라졌습니다. QC를 검토하세요.'
adata = adata[adata.obs.qc_keep].copy()
sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)
assert 'SLC7A5' in adata.var_names, 'QC 후 SLC7A5가 없습니다. 검출 수준을 확인하세요.'
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata.copy()  # log-normalized expression, raw UMI는 layers['counts']

In [ ]:
def cluster_expression(a, key, resolution):
    sc.pp.highly_variable_genes(a, n_top_genes=min(2000, a.n_vars - 1),
                               flavor='seurat', batch_key='donor')
    a.var.loc[a.var_names.isin(['SLC7A5']), 'highly_variable'] = False
    hvg = a[:, a.var.highly_variable].copy()
    sc.pp.scale(hvg, max_value=10)
    n_pcs = min(30, hvg.n_vars - 1, hvg.n_obs - 1)
    sc.tl.pca(hvg, n_comps=n_pcs, random_state=SEED, svd_solver='arpack')
    a.obsm['X_pca'] = hvg.obsm['X_pca'].copy()
    a.uns['pca'] = hvg.uns['pca'].copy()
    # 보정 전 donor UMAP을 진단용으로 유지합니다.
    sc.pp.neighbors(a, n_neighbors=min(15, a.n_obs - 1), n_pcs=n_pcs,
                    use_rep='X_pca', key_added='uncorrected_neighbors', random_state=SEED)
    sc.tl.umap(a, neighbors_key='uncorrected_neighbors', random_state=SEED)
    a.obsm['X_umap_uncorrected'] = a.obsm['X_umap'].copy()
    assert a.obs[BATCH_KEY].nunique() >= 2, 'Harmony에는 여러 donor가 필요합니다.'
    sc.external.pp.harmony_integrate(a, key=BATCH_KEY, basis='X_pca',
        adjusted_basis='X_pca_harmony', theta=HARMONY_THETA,
        max_iter_harmony=HARMONY_MAX_ITER, random_state=SEED,
        nclust=max(2, min(100, round(a.n_obs / 30))))
    assert a.obsm['X_pca_harmony'].shape == a.obsm['X_pca'].shape
    assert np.isfinite(a.obsm['X_pca_harmony']).all()
    a.uns['harmony_configuration'] = {
        'batch_key': BATCH_KEY, 'theta': HARMONY_THETA, 'max_iter_harmony': HARMONY_MAX_ITER,
        'seed': SEED, 'harmonypy_version': hm.__version__, 'neighbors_representation': 'X_pca_harmony'}
    sc.pp.neighbors(a, n_neighbors=min(15, a.n_obs - 1), n_pcs=n_pcs,
                    use_rep='X_pca_harmony', random_state=SEED)
    sc.tl.umap(a, random_state=SEED)
    sc.tl.leiden(a, resolution=resolution, key_added=key, random_state=SEED,
                 flavor='igraph', n_iterations=2, directed=False)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sc.pl.embedding(a, basis='umap_uncorrected', color=BATCH_KEY, ax=axes[0],
                    title='Before Harmony', show=False, legend_loc=None)
    sc.pl.umap(a, color=BATCH_KEY, ax=axes[1], title='After Harmony', show=False)
    fig.tight_layout()
    save_current(f'{key}_donor_before_after_Harmony')
    return a

def expr(a, gene):
    if gene not in a.var_names:
        return np.zeros(a.n_obs)
    x = a[:, gene].X
    return x.toarray().ravel() if sp.issparse(x) else np.asarray(x).ravel()

def save_current(name):
    plt.gcf().savefig(FIGURES / f'{name}.pdf', bbox_inches='tight')
    plt.gcf().savefig(FIGURES / f'{name}.png', dpi=200, bbox_inches='tight')
    plt.show()

def marker_dotplot(a, panels, groupby, filename):
    if a.n_obs == 0:
        # 후보가 없어 새 plot을 만들지 않을 때 이전 후보의 그림이 남지 않도록 보관합니다.
        previous = FIGURES / f'{filename}.pdf'
        if previous.exists():
            archive = ANALYSIS / 'previous_outputs'
            archive.mkdir(exist_ok=True)
            previous.replace(archive / f'{filename}_previous.pdf')
        print(f'{filename}: 선택된 후보가 없어 dotplot을 생략합니다.')
        return
    present = {k: [g for g in v if g in a.var_names] for k, v in panels.items()}
    present = {k: v for k, v in present.items() if v}
    dp = sc.pl.dotplot(a, present, groupby=groupby, use_raw=False,
                      standard_scale='var', return_fig=True)
    dp.savefig(str(FIGURES / f'{filename}.pdf'))
    dp.show()

def score_panels(a, panels, prefix):
    keys = []
    for name, genes in panels.items():
        genes = [g for g in genes if g in a.var_names]
        if len(genes) >= 2:
            key = prefix + name
            sc.tl.score_genes(a, genes, score_name=key, use_raw=False, random_state=SEED)
            keys.append(key)
    return keys

cluster_expression(adata, 'all_cluster', ALL_RESOLUTION)
sc.pl.umap(adata, color=['donor', 'all_cluster'], ncols=2, show=False)
save_current('02_all_UMAP_donor_cluster')

## 3. Marker로 myeloid 후보 선택

여러 lineage signature를 동일한 방식으로 점수화합니다. 자동 선택은 cluster의 평균 myeloid score가
다른 lineage보다 높고 양수인 경우입니다. NK와 공유하는 TYROBP/FCER1G만으로 myeloid를 판단하지 않도록
SPI1/CSF1R/LYZ/LST1 및 경쟁 lineage marker를 함께 검토합니다.
`MYELOID_CLUSTERS`를 수동 지정하면 그 선택을 우선합니다. 점수는 세포형 판정의 통계 검정이 아닙니다.

In [ ]:
LINEAGE = {
    'Myeloid': ['SPI1', 'CSF1R', 'LYZ', 'LST1', 'AIF1', 'CTSS', 'FCER1G', 'TYROBP'],
    'T_cell': ['CD3D', 'CD3E', 'TRAC', 'CD247'],
    'NK': ['NKG7', 'GNLY', 'PRF1', 'KLRD1', 'CTSW'],
    'B_cell': ['MS4A1', 'CD79A', 'CD79B', 'CD37'],
    'Hepatocyte': ['ALB', 'TTR', 'APOA1', 'ASGR1', 'CPS1'],
    'Cholangiocyte': ['KRT19', 'EPCAM', 'KRT7', 'SOX9'],
    'Endothelial': ['PECAM1', 'VWF', 'EMCN', 'KDR', 'CLEC4G'],
    'Stellate': ['COL1A1', 'COL1A2', 'DCN', 'LUM', 'PDGFRB'],
}
lineage_keys = score_panels(adata, LINEAGE, 'lineage_')
lineage_summary = adata.obs.groupby('all_cluster', observed=True)[lineage_keys].mean()
lineage_summary['dominant_signature'] = lineage_summary[lineage_keys].idxmax(axis=1).str.replace('lineage_', '')
lineage_summary['n_cells'] = adata.obs.all_cluster.value_counts()
lineage_summary.to_csv(ANALYSIS / 'all_cluster_lineage_scores.csv')
display(lineage_summary)
marker_dotplot(adata, LINEAGE, 'all_cluster', '03_all_lineage_markers')
automatic_myeloid = lineage_summary.index[(lineage_summary.dominant_signature == 'Myeloid')
                                          & (lineage_summary.lineage_Myeloid > 0)].tolist()
selected_myeloid = automatic_myeloid if MYELOID_CLUSTERS is None else list(map(str, MYELOID_CLUSTERS))
assert set(selected_myeloid).issubset(set(adata.obs.all_cluster)), 'Myeloid cluster ID 확인'
adata.obs['myeloid_candidate'] = adata.obs.all_cluster.isin(selected_myeloid)
adata.obs['lineage_candidate'] = adata.obs.all_cluster.map(lineage_summary.dominant_signature).astype(str)
print('Selected myeloid candidate clusters:', selected_myeloid)
display(pd.crosstab(adata.obs.donor, adata.obs.myeloid_candidate).reindex(DONORS, fill_value=0))
donor_mix = pd.crosstab(adata.obs.all_cluster, adata.obs.donor).reindex(columns=DONORS, fill_value=0)
donor_mix.to_csv(ANALYSIS / 'all_cluster_donor_counts.csv')
sc.pl.umap(adata, color=['lineage_candidate', 'myeloid_candidate', 'SLC7A5'], ncols=3, show=False)
save_current('04_all_lineage_SLC7A5')
assert adata.obs.myeloid_candidate.sum() >= 30, 'Myeloid 후보가 너무 적습니다. 전체 marker와 cluster 선택을 확인하세요.'
myeloid = adata[adata.obs.myeloid_candidate].copy()

## 4. Myeloid 재클러스터링 및 monocyte 후보

Figure 2 marker 구성을 참조하되 `Special_monocyte`를 자동 주석의 선택지에 넣지 않습니다.
Classical/non-classical monocyte와 Kupffer/LAM/DC를 먼저 구분합니다.
Monocyte 후보는 monocyte signature가 우세한 cluster로 정의하고, 이후 SLC7A5 발현 분할을 별도로 수행합니다.
SLC7A5를 제외한 PPIF/ANPEP/ITGAX/VCAN signature는 기존 특성과의 일치도를 확인하는 보조 지표입니다.
재클러스터링 후 canonical myeloid marker 6개 중 3개 이상이 cluster 핵의 10% 이상에서 검출되는지
검토하여 부족한 cluster를 제외합니다. 이는 명시적 탐색적 QC 기준이며 수동 선택으로 바꿀 수 있습니다.
Classical 후보에서 CD14/FCN1/VCAN 모두 10% 미만, 또는 non-classical 후보에서 FCGR3A가 10% 미만
검출되면 `Monocyte_DC_ambiguous`로 표시합니다. 이 기준은 확정 주석 기준이 아니라 검토 필요 표시입니다.
Myeloid에서도 HVG/PCA를 새로 계산하고 donor Harmony를 다시 적용합니다. 전체 세포에서 얻은 보정 좌표를 그대로 재사용하지 않습니다.
재클러스터링 후에도 경쟁 lineage score가 myeloid보다 우세한 cluster는 자동 선택에서 제외합니다.
모든 subtype score가 0 이하인 cluster는 Unresolved로 남기고, 단순히 가장 큰 점수가 monocyte라는 이유로 monocyte에 포함하지 않습니다.

In [ ]:
MYELOID_MARKERS = {
    'Classical_monocyte': ['S100A9', 'S100A8', 'S100A6', 'LYZ', 'FCN1', 'CD14', 'VCAN'],
    'Nonclassical_monocyte': ['IFITM2', 'IFITM3', 'CD52', 'LST1', 'FCGR3A'],
    'Kupffer_cell': ['CD5L', 'CD163', 'SLC40A1', 'MARCO', 'VCAM1'],
    'LAM': ['TREM2', 'LGALS3', 'C1QC', 'APOC1', 'GPNMB'],
    'Macrophage_C1Q': ['C1QA', 'C1QB', 'C1QC', 'AIF1', 'APOE'],
    'cDC1': ['SNX3', 'DNASE1L3', 'CLEC9A', 'IRF8', 'IDO1'],
    'cDC2': ['HLA-DQB1', 'HLA-DPB1', 'HLA-DQA1', 'CD1C', 'FCER1A'],
    'Migratory_DC': ['GBP1', 'IL4I1', 'CXCL10', 'LAMP3', 'CD83'],
}
cluster_expression(myeloid, 'myeloid_cluster', MYELOID_RESOLUTION)
canonical = [g for g in ['SPI1', 'CSF1R', 'LST1', 'CTSS', 'FCER1G', 'TYROBP'] if g in myeloid.var_names]
canonical_detection = pd.DataFrame({g: expr(myeloid, g) > 0 for g in canonical}, index=myeloid.obs_names)
canonical_detection['cluster'] = myeloid.obs.myeloid_cluster.to_numpy()
canonical_audit = canonical_detection.groupby('cluster', observed=True).mean()
canonical_audit['n_markers_detected_in_10pct'] = (canonical_audit[canonical] >= 0.10).sum(axis=1)
canonical_audit['n_cells'] = myeloid.obs.myeloid_cluster.value_counts()
# Myeloid 전체 cluster에 섞인 간세포 등은 재클러스터링 후 경쟁 lineage도 확인합니다.
recluster_lineage = myeloid.obs.groupby('myeloid_cluster', observed=True)[lineage_keys].mean()
canonical_audit['dominant_lineage'] = recluster_lineage.idxmax(axis=1).str.replace('lineage_', '')
canonical_audit['mean_lineage_Myeloid'] = recluster_lineage['lineage_Myeloid']
auto_keep = canonical_audit.index[(canonical_audit.n_markers_detected_in_10pct >= 3)
    & (canonical_audit.dominant_lineage == 'Myeloid')
    & (canonical_audit.mean_lineage_Myeloid > 0)].tolist()
keep_reclusters = auto_keep if MYELOID_KEEP_RECLUSTERS is None else list(map(str, MYELOID_KEEP_RECLUSTERS))
assert set(keep_reclusters).issubset(set(myeloid.obs.myeloid_cluster))
canonical_audit['retained'] = canonical_audit.index.isin(keep_reclusters)
canonical_audit.to_csv(ANALYSIS / 'myeloid_recluster_marker_QC.csv')
display(canonical_audit)
excluded_ids = myeloid.obs_names[~myeloid.obs.myeloid_cluster.isin(keep_reclusters)]
adata.obs.loc[excluded_ids, 'myeloid_candidate'] = False
adata.obs['lineage_candidate'] = adata.obs.lineage_candidate.astype(str)
adata.obs.loc[excluded_ids, 'lineage_candidate'] = 'Excluded_from_myeloid_marker_review'
print('Excluded after myeloid marker review:', len(excluded_ids))
myeloid = myeloid[myeloid.obs.myeloid_cluster.isin(keep_reclusters)].copy()
assert myeloid.n_obs >= 30, 'Marker-supported myeloid 후보가 부족합니다. QC를 검토하세요.'
subtype_keys = score_panels(myeloid, MYELOID_MARKERS, 'subtype_')
subtype_summary = myeloid.obs.groupby('myeloid_cluster', observed=True)[subtype_keys].mean()
subtype_summary['dominant_signature'] = subtype_summary[subtype_keys].idxmax(axis=1).str.replace('subtype_', '')
subtype_summary['n_cells'] = myeloid.obs.myeloid_cluster.value_counts()
subtype_summary['annotation_candidate'] = subtype_summary.dominant_signature
weak_all = subtype_summary[subtype_keys].max(axis=1) <= 0
subtype_summary.loc[weak_all, 'annotation_candidate'] = 'Unresolved_myeloid'
mono_check = ['CD14', 'FCN1', 'VCAN', 'FCGR3A']
mono_detection = pd.DataFrame({g: expr(myeloid, g) > 0 for g in mono_check}, index=myeloid.obs_names)
mono_detection['cluster'] = myeloid.obs.myeloid_cluster.to_numpy()
mono_detection = mono_detection.groupby('cluster', observed=True).mean()
mono_detection.to_csv(ANALYSIS / 'myeloid_cluster_monocyte_marker_detection.csv')
is_mono_signature = subtype_summary.dominant_signature.isin(['Classical_monocyte', 'Nonclassical_monocyte'])
weak_classical = (subtype_summary.dominant_signature == 'Classical_monocyte') & (
    mono_detection[['CD14', 'FCN1', 'VCAN']].max(axis=1) < 0.10)
weak_nonclassical = (subtype_summary.dominant_signature == 'Nonclassical_monocyte') & (
    mono_detection.FCGR3A < 0.10)
weak_mono = (weak_classical | weak_nonclassical) & ~weak_all
subtype_summary.loc[weak_mono, 'annotation_candidate'] = 'Monocyte_DC_ambiguous'
subtype_summary['monocyte_marker_support_weak'] = weak_mono
subtype_summary.to_csv(ANALYSIS / 'myeloid_cluster_signature_scores.csv')
display(subtype_summary)
marker_dotplot(myeloid, {**MYELOID_MARKERS,
    'SLC7A5_high_reference': ['PPIF', 'ANPEP', 'SLC7A5', 'ITGAX', 'VCAN']},
    'myeloid_cluster', '05_myeloid_marker_dotplot')
auto_mono = subtype_summary.index[subtype_summary.dominant_signature.isin(
                                  ['Classical_monocyte', 'Nonclassical_monocyte']) & ~weak_all].tolist()
selected_mono = auto_mono if MONOCYTE_CLUSTERS is None else list(map(str, MONOCYTE_CLUSTERS))
assert set(selected_mono).issubset(set(myeloid.obs.myeloid_cluster)), 'Monocyte cluster ID 확인'
myeloid.obs['subtype_candidate'] = myeloid.obs.myeloid_cluster.map(subtype_summary.annotation_candidate).astype(str)
myeloid.obs['monocyte_candidate'] = myeloid.obs.myeloid_cluster.isin(selected_mono)
sc.tl.score_genes(myeloid, [g for g in ['PPIF', 'ANPEP', 'ITGAX', 'VCAN'] if g in myeloid.var_names],
                  score_name='reference_signature_without_SLC7A5', use_raw=False, random_state=SEED)
myeloid.obs['SLC7A5_log1p'] = expr(myeloid, 'SLC7A5')
sc.pl.umap(myeloid, color=['donor', 'myeloid_cluster', 'subtype_candidate', 'SLC7A5',
                           'reference_signature_without_SLC7A5', 'monocyte_candidate'], ncols=3, show=False)
save_current('06_myeloid_UMAP')
print('Selected monocyte candidate clusters:', selected_mono)
if subtype_summary.loc[selected_mono, 'monocyte_marker_support_weak'].any():
    print('주의: 선택된 후보에 monocyte/DC 구분이 불명확한 cluster가 있습니다. 확정 monocyte 주석으로 사용하지 마세요.')
# Cluster marker ranking is descriptive: no cell-level P values are exported or used for inference.
sc.tl.rank_genes_groups(myeloid, groupby='myeloid_cluster', method='t-test',
                       use_raw=False, n_genes=30, key_added='cluster_marker_ranking')
ranked = sc.get.rank_genes_groups_df(myeloid, group=None, key='cluster_marker_ranking')
ranked[['group', 'names', 'logfoldchanges']].to_csv(ANALYSIS / 'myeloid_top30_markers_descriptive.csv', index=False)
mix = pd.crosstab(myeloid.obs.myeloid_cluster, myeloid.obs.donor).reindex(columns=DONORS, fill_value=0)
mix.to_csv(ANALYSIS / 'myeloid_cluster_donor_counts.csv')
display(mix)
if myeloid.obs.monocyte_candidate.sum() < 20:
    print('Monocyte 후보가 20개 미만입니다. marker와 MONOCYTE_CLUSTERS를 검토하세요. 부족한 후보를 임의로 채우지 않습니다.')
monocytes = myeloid[myeloid.obs.monocyte_candidate].copy()

## 4.5. scRNA-seq SLC7A5-high monocyte DEG와 MASL/MASH PBMC cluster 비교

기존 `scmono5_DEG.csv`의 mono5/SLC7A5-high **vs rest** DEG를 주 참조로 사용합니다.
`MASLD_mono_DEG_allclusters.csv`의 `SLC7A5_high_monocyte` 효과값과 일치하는지 확인하고 검출률을 함께 사용합니다.
기존 Figure 2의 Special vs Classical DEG는 별도 민감도 비교로 사용합니다.

PBMC에서는 **donor × cluster별 raw UMI를 합산**하고, 같은 donor의 나머지 myeloid와 비교한
`log2((CPM_cluster + 1)/(CPM_rest + 1))`를 계산합니다. donor의 평균 효과와 참조 DEG의 signed logFC를
동일한 유전자 순서로 정렬해 Spearman correlation을 계산합니다. 유전자 간 상관의 P 값은 생물학적 검정으로 사용하지 않습니다.

- 참조 BH < 0.05, |logFC| ≥ 0.5, 참조 어느 쪽에서든 검출률 ≥ 5%, PBMC 전체 검출률 ≥ 1%의 공통 유전자를 사용합니다.
- SLC7A5 자체와 MT-/RPS/RPL prefix 유전자는 matching에서 제외합니다. SLC7A5 및 PPIF/ANPEP/ITGAX/VCAN은 별도 marker panel로 확인합니다.
- MASL/MASH PBMC cluster(0–7)의 전체 효과 profile, donor별 상관 및 donor를 하나씩 제외한 결과를 저장합니다.
- 주 참조와 가장 높은 **양의** 상관을 보이는 cluster를 `SLC7A5_high_monocyte`로 주석합니다.
  이는 참조 기반 탐색적 주석이며, 상관계수의 절대 크기와 marker를 함께 해석해야 합니다.
- 이후 주 분석의 high 집단은 **주석된 cluster 전체**입니다. 기존 양성값 75백분위 발현 분할은 별도 보조 컬럼에 보존합니다.

[Spearman correlation 문서](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.spearmanr.html)

선택은 monocyte marker 지지가 있는 cluster 중 참조 상관이 가장 높은 양의 cluster입니다. Condition별 빈도는 선택에 사용하지 않습니다.


In [ ]:
assert DATASET == 'pbmc', '이 reference matching은 PBMC 섹션에서 실행합니다.'
REFERENCE_MONO5_DEG = Path(dataset_path('hwangbj_reference_mono5'))
REFERENCE_ALL_DEG = Path(dataset_path('hwangbj_reference_all'))
REFERENCE_CLASSICAL_DEG = Path(dataset_path('hwangbj_reference_classical'))
REFERENCE_CLUSTER_OVERRIDE = None  # None: 양의 상관이 가장 높은 cluster를 선택; 수동 지정 예: '3'
REF_BH_MAX = 0.05
REF_MIN_ABS_LFC = 0.5
REF_MIN_DETECTION = 0.05
PBMC_MIN_DETECTION = 0.01
MATCHING_DIR = ANALYSIS / 'reference_matching'
MATCHING_DIR.mkdir(exist_ok=True)
cluster_ids = sorted(myeloid.obs.myeloid_cluster.astype(str).unique(), key=int)
print('PBMC myeloid cluster IDs:', cluster_ids, '| n clusters:', len(cluster_ids))

reference_main = pd.read_csv(REFERENCE_MONO5_DEG).set_index('names')
reference_all = pd.read_csv(REFERENCE_ALL_DEG)
reference_aux = reference_all.loc[reference_all.group.eq('SLC7A5_high_monocyte')].set_index('names')
assert reference_main.index.is_unique and reference_aux.index.is_unique
assert len(reference_aux) and reference_main.index.isin(reference_aux.index).all()
np.testing.assert_allclose(reference_main.logfoldchanges,
                           reference_aux.reindex(reference_main.index).logfoldchanges)
reference_main = reference_main.join(reference_aux[['pct_nz_group', 'pct_nz_reference']])
reference_classical = pd.read_csv(REFERENCE_CLASSICAL_DEG).set_index('names')
assert reference_classical.index.is_unique
reference_tables = {'SLC7A5-high vs rest': reference_main,
                    'SLC7A5-high vs Classical': reference_classical}
primary_reference = 'SLC7A5-high vs rest'
reference_sources = {str(path): hashlib.sha256(path.read_bytes()).hexdigest()
                     for path in [REFERENCE_MONO5_DEG, REFERENCE_ALL_DEG, REFERENCE_CLASSICAL_DEG]}

In [ ]:
# Raw UMI pseudobulk: every nucleus/cell belongs to exactly one donor × cluster row.
pb_counts_rows, pb_rows, pb_effects = [], [], {}
counts = myeloid.layers['counts']
cluster_labels = myeloid.obs.myeloid_cluster.astype(str)
for donor in DONORS:
    donor_mask = (myeloid.obs.donor == donor).to_numpy()
    donor_total = np.asarray(counts[donor_mask].sum(axis=0, dtype=np.float64)).ravel()
    for cluster in cluster_ids:
        mask = donor_mask & cluster_labels.eq(cluster).to_numpy()
        n_group, n_rest = int(mask.sum()), int(donor_mask.sum() - mask.sum())
        group_sum = np.asarray(counts[mask].sum(axis=0, dtype=np.float64)).ravel()
        rest_sum = donor_total - group_sum
        row_id = f'{donor}__cluster{cluster}'
        pb_rows.append({'pseudobulk_id': row_id, 'donor': donor, 'cluster': cluster,
                        'n_cells': n_group, 'n_rest': n_rest,
                        'eligible_for_DE': n_group >= MIN_CELLS_PSEUDOBULK and n_rest >= MIN_CELLS_PSEUDOBULK})
        pb_counts_rows.append(sp.csr_matrix(group_sum.astype(np.int64).reshape(1, -1)))
        if n_group and n_rest and group_sum.sum() > 0 and rest_sum.sum() > 0:
            group_cpm = 1e6 * group_sum / group_sum.sum()
            rest_cpm = 1e6 * rest_sum / rest_sum.sum()
            pb_effects[(cluster, donor)] = np.log2((group_cpm + 1) / (rest_cpm + 1))
        else:
            pb_effects[(cluster, donor)] = np.full(myeloid.n_vars, np.nan)

cluster_donor_metadata = pd.DataFrame(pb_rows)
cluster_donor_counts = sp.vstack(pb_counts_rows, format='csr')
assert int(cluster_donor_counts.sum()) == int(counts.sum(dtype=np.float64))
sp.save_npz(MATCHING_DIR / 'PBMC_cluster_donor_pseudobulk_counts.npz', cluster_donor_counts)
cluster_donor_metadata.to_csv(MATCHING_DIR / 'PBMC_cluster_donor_pseudobulk_metadata.csv', index=False)
pd.Series(myeloid.var_names, name='gene').to_csv(MATCHING_DIR / 'PBMC_pseudobulk_genes.csv', index=False)
donor_effects = pd.DataFrame(pb_effects, index=myeloid.var_names)
donor_effects.columns = pd.MultiIndex.from_tuples(donor_effects.columns, names=['cluster', 'donor'])
cluster_effects = pd.DataFrame({c: donor_effects[c].mean(axis=1) for c in cluster_ids})
cluster_effects.index.name = 'gene'
cluster_effects.to_csv(MATCHING_DIR / 'PBMC_cluster_vs_rest_donor_mean_log2CPM_effects.csv')
donor_effects.to_csv(MATCHING_DIR / 'PBMC_cluster_vs_rest_individual_donor_effects.csv')
pbmc_detection = pd.Series(np.asarray((myeloid.X > 0).mean(axis=0)).ravel(), index=myeloid.var_names)

# Descriptive DEG effect profiles for all joint-cohort clusters; no cell-level P values.
deg_profiles = []
for cluster in cluster_ids:
    mask = cluster_labels.eq(cluster).to_numpy()
    table = pd.DataFrame({'gene': myeloid.var_names, 'cluster': cluster,
        'mean_log2CPM_difference': cluster_effects[cluster].to_numpy(),
        'n_valid_donors': donor_effects[cluster].notna().sum(axis=1).to_numpy(),
        'pct_detected_cluster': np.asarray((myeloid.X[mask] > 0).mean(axis=0)).ravel(),
        'pct_detected_rest': np.asarray((myeloid.X[~mask] > 0).mean(axis=0)).ravel()})
    deg_profiles.append(table)
pd.concat(deg_profiles, ignore_index=True).to_csv(MATCHING_DIR / 'PBMC_cluster_DEG_effect_profiles.csv', index=False)
display(cluster_donor_metadata)

In [ ]:
correlations = pd.DataFrame(index=reference_tables, columns=cluster_ids, dtype=float)
gene_sets, aligned_profiles = {}, {}
for label, reference in reference_tables.items():
    detection_ok = pbmc_detection.index[pbmc_detection >= PBMC_MIN_DETECTION]
    genes = reference.index.intersection(detection_ok)
    table = reference.loc[genes].copy()
    mask = ((table.pvals_adj < REF_BH_MAX) & (table.logfoldchanges.abs() >= REF_MIN_ABS_LFC)
            & (table[['pct_nz_group', 'pct_nz_reference']].max(axis=1) >= REF_MIN_DETECTION))
    genes = table.index[mask]
    genes = genes[~genes.str.startswith(('MT-', 'RPS', 'RPL')) & (genes != 'SLC7A5')]
    genes = genes[cluster_effects.reindex(genes).notna().all(axis=1)]
    assert len(genes) >= 50, f'{label}: 공통 informative 유전자가 부족합니다.'
    gene_sets[label] = genes
    profile = cluster_effects.loc[genes].copy()
    profile.insert(0, label, table.loc[genes, 'logfoldchanges'])
    aligned_profiles[label] = profile
    correlations.loc[label] = profile[cluster_ids].corrwith(profile[label], method='spearman')
    slug = 'vs_rest' if label == primary_reference else 'vs_classical'
    profile.to_csv(MATCHING_DIR / f'aligned_reference_PBMC_effects_{slug}.csv')
    table.loc[genes].to_csv(MATCHING_DIR / f'scRNA_reference_selected_DEG_{slug}.csv')
correlations.to_csv(MATCHING_DIR / 'reference_DEG_cluster_Spearman_correlations.csv')
display(correlations)

with plt.rc_context({'font.size': 10, 'axes.labelsize': 10, 'xtick.labelsize': 9, 'ytick.labelsize': 9}):
    fig, ax = plt.subplots(figsize=(11, 3.4))
    sns.heatmap(correlations, cmap='vlag', center=0, vmin=-1, vmax=1, annot=True, fmt='.3f',
                linewidths=.5, ax=ax, cbar_kws={'label': 'Spearman rho'})
    ax.set_yticklabels([f'{label} (n={len(gene_sets[label])} genes)' for label in correlations.index], rotation=0)
    ax.set_xticklabels(cluster_ids, rotation=0, fontsize=10)
    ax.set_xlabel('PBMC myeloid cluster')
    ax.set_title('scRNA-seq reference DEG vs PBMC cluster effects')
    fig.tight_layout()
    save_current('11_PBMC_reference_DEG_correlation_heatmap')
    matrix = aligned_profiles[primary_reference].corr(method='spearman')
    matrix.to_csv(MATCHING_DIR / 'reference_and_PBMC_profile_Spearman_matrix.csv')
    fig, ax = plt.subplots(figsize=(9, 7.5))
    sns.heatmap(matrix, cmap='vlag', center=0, vmin=-1, vmax=1, annot=True, fmt='.2f',
                ax=ax, cbar_kws={'label': 'Spearman rho'})
    ax.set_title('Shared DEG effect profile correlations (SLC7A5 excluded)')
    fig.tight_layout()
    save_current('12_PBMC_reference_and_cluster_correlation_matrix')

primary_genes = gene_sets[primary_reference]
primary_vector = reference_main.loc[primary_genes, 'logfoldchanges']
donor_correlations = pd.DataFrame(index=DONORS, columns=cluster_ids, dtype=float)
leave_one_out = donor_correlations.copy()
for donor in DONORS:
    for cluster in cluster_ids:
        donor_correlations.loc[donor, cluster] = donor_effects.loc[primary_genes, (cluster, donor)].corr(primary_vector, method='spearman')
        other_donors = [d for d in DONORS if d != donor]
        loo_vector = donor_effects[cluster].loc[primary_genes, other_donors].mean(axis=1)
        leave_one_out.loc[donor, cluster] = loo_vector.corr(primary_vector, method='spearman')
donor_correlations.to_csv(MATCHING_DIR / 'individual_donor_reference_correlations.csv')
leave_one_out.to_csv(MATCHING_DIR / 'leave_one_donor_out_reference_correlations.csv')
with plt.rc_context({'font.size': 10, 'xtick.labelsize': 9, 'ytick.labelsize': 9}):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, frame, title in zip(axes, [donor_correlations, leave_one_out],
                                ['Individual donors', 'Leave one donor out']):
        sns.heatmap(frame, cmap='vlag', center=0, vmin=-1, vmax=1, annot=True, fmt='.2f', ax=ax,
                    cbar_kws={'label': 'Spearman rho'})
        ax.set_title(title)
        ax.set_xlabel('PBMC cluster')
    axes[1].set_ylabel('Excluded donor')
    fig.tight_layout()
    save_current('13_PBMC_donor_reference_similarity_heatmap')

In [ ]:
primary_scores = correlations.loc[primary_reference].dropna().sort_values(ascending=False)
eligible_reference_clusters = subtype_summary.index[subtype_summary.annotation_candidate.isin(['Classical_monocyte', 'Nonclassical_monocyte'])].astype(str).tolist()
eligible_scores = primary_scores.loc[primary_scores.index.isin(eligible_reference_clusters)]
automatic_reference_cluster = str(eligible_scores.index[0]) if len(eligible_scores) and eligible_scores.iloc[0] > 0 else None
selected_reference_cluster = (automatic_reference_cluster if REFERENCE_CLUSTER_OVERRIDE is None
                               else str(REFERENCE_CLUSTER_OVERRIDE))
if selected_reference_cluster is not None:
    assert selected_reference_cluster in eligible_reference_clusters, 'Monocyte marker 지지가 있는 cluster를 선택하세요.'
    assert primary_scores[selected_reference_cluster] > 0, '양의 참조 유사도가 없는 cluster를 high로 지정하지 않습니다.'
base_subtypes = myeloid.obs.get('subtype_before_reference_match', myeloid.obs.subtype_candidate).astype(str)
myeloid.obs['subtype_before_reference_match'] = base_subtypes
myeloid.obs['celltype_refined'] = base_subtypes.copy()
myeloid.obs['SLC7A5_reference_cluster'] = (cluster_labels.eq(selected_reference_cluster)
                                          if selected_reference_cluster is not None else False)
myeloid.obs.loc[myeloid.obs.SLC7A5_reference_cluster, 'celltype_refined'] = 'SLC7A5_high_monocyte'
myeloid.obs['subtype_candidate'] = myeloid.obs.celltype_refined.astype(str)
myeloid.obs['annotation_method'] = 'marker-based provisional'
myeloid.obs.loc[myeloid.obs.SLC7A5_reference_cluster, 'annotation_method'] = 'reference_DEG_correlation_provisional'
adata.obs['celltype_refined'] = adata.obs.lineage_candidate.astype(str)
adata.obs.loc[myeloid.obs_names, 'celltype_refined'] = myeloid.obs.celltype_refined.to_numpy()
adata.obs['SLC7A5_reference_cluster'] = False
adata.obs.loc[myeloid.obs_names, 'SLC7A5_reference_cluster'] = myeloid.obs.SLC7A5_reference_cluster.to_numpy()
monocytes = myeloid[myeloid.obs.monocyte_candidate].copy()
matched_rho = float(primary_scores[selected_reference_cluster]) if selected_reference_cluster is not None else None
reference_match = {
    'selected_cluster': selected_reference_cluster, 'automatic_best_cluster': automatic_reference_cluster,
    'annotation': 'SLC7A5_high_monocyte', 'annotation_status': 'reference-matched provisional',
    'selection_method': 'highest positive signed-DEG Spearman correlation among marker-supported monocyte clusters',
    'eligible_monocyte_clusters': eligible_reference_clusters,
    'selection_uses_condition_frequency_or_P_value': False,
    'n_myeloid_clusters': len(cluster_ids),
    'cohort_donors': DONORS,
    'primary_reference': primary_reference, 'primary_rho': matched_rho,
    'n_primary_genes': len(primary_genes), 'SLC7A5_excluded_from_matching': True,
    'donor_best_clusters': donor_correlations[eligible_reference_clusters].idxmax(axis=1).to_dict(),
    'leave_one_donor_out_best_clusters': leave_one_out[eligible_reference_clusters].idxmax(axis=1).to_dict(),
    'reference_sources_sha256': reference_sources,
    'primary_group_definition': 'whole reference-matched cluster; not expression quantile',
    'n_annotated_cells': int(myeloid.obs.SLC7A5_reference_cluster.sum()),
    'correlation_is_biological_hypothesis_test': False,
}
if matched_rho is not None and matched_rho < .3:
    print(f'참조 상관 rho={matched_rho:.3f}는 높지 않습니다. 상대적 최상위 cluster의 탐색적 주석으로 해석하세요.')
(MATCHING_DIR / 'reference_cluster_annotation.json').write_text(json.dumps(reference_match, indent=2))
display(reference_match)
marker_panels = {'Reference': ['SLC7A5', 'PPIF', 'ANPEP', 'ITGAX', 'VCAN'],
                 'Monocyte': ['CD14', 'FCN1', 'S100A8', 'S100A9', 'FCGR3A'],
                 'DC/macrophage check': ['CD1C', 'FCER1A', 'CLEC9A', 'C1QA', 'MARCO', 'TREM2']}
marker_dotplot(myeloid, marker_panels, 'myeloid_cluster', '14_PBMC_reference_marker_check_by_cluster')
sc.pl.umap(myeloid, color=['myeloid_cluster', 'celltype_refined', 'Condition', 'SLC7A5',
                           'reference_signature_without_SLC7A5'], ncols=2, show=False)
save_current('15_PBMC_reference_annotation_UMAP')
myeloid.obs.to_csv(MATCHING_DIR / 'PBMC_reference_annotated_cell_metadata.csv')
marker_rows = []
for donor in DONORS:
    for cluster in cluster_ids:
        mask = (myeloid.obs.donor == donor) & cluster_labels.eq(cluster)
        for gene in ['SLC7A5', 'PPIF', 'ANPEP', 'ITGAX', 'VCAN']:
            values = expr(myeloid[mask], gene)
            marker_rows.append({'donor': donor, 'cluster': cluster, 'gene': gene, 'n_cells': len(values),
                'mean_log1p': float(values.mean()) if len(values) else np.nan,
                'detected_fraction': float((values > 0).mean()) if len(values) else np.nan})
pd.DataFrame(marker_rows).to_csv(MATCHING_DIR / 'PBMC_donor_cluster_marker_expression.csv', index=False)
selected_profile = pd.concat(deg_profiles, ignore_index=True)
selected_profile = selected_profile.loc[selected_profile.cluster.eq(selected_reference_cluster)]
selected_profile.to_csv(MATCHING_DIR / 'SLC7A5_reference_cluster_vs_other_myeloid_effects.csv', index=False)
# Downstream pathway tools may use this donor-average effect ranking; no inferred cell-level P values.
selected_profile[['gene', 'mean_log2CPM_difference']].dropna().sort_values('mean_log2CPM_difference', ascending=False).to_csv(
    MATCHING_DIR / 'SLC7A5_reference_cluster_prerank.rnk', sep='	', index=False, header=False)
# Map the old MASL-only cluster 3 to the new joint-cohort clusters using shared barcodes.
legacy_path = Path(dataset_path('hwangbj_legacy_masl_myeloid', required=False))
if legacy_path.exists():
    legacy_object = ad.read_h5ad(legacy_path, backed='r')
    legacy_obs = legacy_object.obs.copy()
    legacy_object.file.close()
    shared = myeloid.obs_names.intersection(legacy_obs.index)
    correspondence = pd.crosstab(myeloid.obs.loc[shared, 'myeloid_cluster'].astype(str),
                                legacy_obs.loc[shared, 'celltype_refined'].astype(str))
    correspondence.to_csv(MATCHING_DIR / 'MASL_only_annotation_to_joint_clusters.csv')
    display(correspondence)


## 5. Reference cluster 주석과 SLC7A5 발현 보조 gate

PBMC 주 분석은 위에서 DEG correlation으로 선택한 **cluster 전체**를 SLC7A5_high_monocyte로 사용합니다.
`SLC7A5_high_candidate`와 `SLC_expression_group`는 이 참조 주석을 반영하며, donor 비율과 pseudobulk도 같은 정의를 사용합니다.
발현 양성값 75백분위 기준은 `SLC7A5_expression_high_q75`에 따로 보존합니다.
CD14-high / FCGR3A-low / SLC7A5-expression-high triple gate는 탐색적 보조 지표이며 cluster 주석을 대체하지 않습니다.


In [ ]:
slc = expr(monocytes, 'SLC7A5')
positive = slc[slc > 0]
expression_cutoff_available = len(positive) >= MIN_POSITIVE_FOR_CUTOFF
high_available = selected_reference_cluster is not None
slc_cutoff = float(np.quantile(positive, SLC_POSITIVE_QUANTILE)) if expression_cutoff_available else None
monocytes.obs['SLC7A5_detected'] = slc > 0
monocytes.obs['SLC7A5_expression_high_q75'] = ((slc >= slc_cutoff) & (slc > 0)
                                              if expression_cutoff_available else False)
monocytes.obs['SLC7A5_high_candidate'] = monocytes.obs.SLC7A5_reference_cluster.to_numpy()
monocytes.obs['SLC_expression_group'] = np.where(monocytes.obs.SLC7A5_high_candidate,
    'SLC7A5_high_candidate', 'Other_monocytes') if high_available else np.where(slc > 0,
    'detected_only_insufficient_for_high', 'Undetected')
cd14, fcgr3a = expr(monocytes, 'CD14'), expr(monocytes, 'FCGR3A')
cd14_positive = cd14[cd14 > 0]
gate_available = expression_cutoff_available and len(cd14_positive) >= MIN_POSITIVE_FOR_CUTOFF
cd14_cutoff = float(np.quantile(cd14_positive, 0.75)) if gate_available else None
fcgr3a_cutoff = float(np.quantile(fcgr3a, 0.25)) if gate_available else None
monocytes.obs['triple_gate_candidate'] = (
    monocytes.obs.SLC7A5_expression_high_q75 & (cd14 >= cd14_cutoff) & (fcgr3a <= fcgr3a_cutoff)
) if gate_available else False
definition = {'group': 'whole reference-matched SLC7A5_high_monocyte cluster; provisional annotation',
    'annotation_method': 'reference_DEG_correlation', 'selected_reference_cluster': selected_reference_cluster,
    'reference_match': reference_match, 'expression_q75_is_primary_definition': False,
    'monocyte_candidate_n': monocytes.n_obs, 'monocyte_definition_available': bool(monocytes.n_obs),
    'quantile_among_positive_monocytes': SLC_POSITIVE_QUANTILE,
    'minimum_positive_n': MIN_POSITIVE_FOR_CUTOFF, 'positive_n': len(positive),
    'high_definition_available': bool(high_available), 'SLC7A5_log1p_cutoff': slc_cutoff,
    'triple_gate_available': bool(gate_available), 'CD14_positive_q75': cd14_cutoff,
    'FCGR3A_all_q25': fcgr3a_cutoff,
    'myeloid_clusters': selected_myeloid, 'monocyte_clusters': selected_mono}
definition['retained_myeloid_reclusters'] = keep_reclusters
definition['weak_monocyte_marker_clusters'] = subtype_summary.index[
    subtype_summary.monocyte_marker_support_weak].tolist()
(ANALYSIS / 'SLC7A5_candidate_definition.json').write_text(json.dumps(definition, indent=2))
print(json.dumps(definition, indent=2))
monocytes.obs.to_csv(ANALYSIS / 'monocyte_cell_metadata.csv')
for column in ['SLC7A5_detected', 'SLC7A5_high_candidate', 'SLC7A5_expression_high_q75', 'triple_gate_candidate']:
    myeloid.obs[column] = False
    myeloid.obs.loc[monocytes.obs_names, column] = monocytes.obs[column].to_numpy()
sc.pl.umap(myeloid, color=['SLC7A5', 'SLC7A5_high_candidate', 'triple_gate_candidate'], ncols=3, show=False)
save_current('07_SLC7A5_candidates_UMAP')
marker_dotplot(monocytes, {'Monocyte': ['CD14', 'FCGR3A', 'FCN1', 'S100A8', 'S100A9'],
    'Reference': ['PPIF', 'ANPEP', 'SLC7A5', 'ITGAX', 'VCAN'],
    'Macrophage_DC_check': ['C1QA', 'MARCO', 'TREM2', 'CD1C', 'CLEC9A']},
    'SLC_expression_group', '08_SLC7A5_candidate_marker_check')

## 6. Donor별 세포 수·비율 및 signature

각 donor의 high 후보 수를 해당 donor의 **전체 monocyte 후보 수**로 나눈 비율을 주 지표로 표시합니다.
전체 QC 통과 핵과 myeloid 후보를 분모로 한 비율도 함께 저장합니다. high 후보가 없는 donor도 0으로 포함합니다.
그림의 `n`은 donor별 포착된 핵 수이며 생물학적 반복 수는 donor 8명입니다.
비율은 이 데이터에서 회수된 핵의 조성으로 해석합니다.
PBMC의 high 분자는 통합 코호트에서 다시 선택한 reference-matched cluster 전체 세포 수입니다. Expression-q75 gate와 별도로 취급합니다.

In [ ]:
def donor_n(a, mask=None):
    obs = a.obs if mask is None else a.obs.loc[mask]
    return obs.donor.value_counts().reindex(DONORS, fill_value=0).astype(int)

fractions = pd.DataFrame({'total_QC_n': donor_n(adata), 'myeloid_n': donor_n(myeloid),
    'monocyte_n': donor_n(monocytes), 'SLC7A5_detected_n': donor_n(monocytes, monocytes.obs.SLC7A5_detected),
    'high_n': donor_n(monocytes, monocytes.obs.SLC7A5_high_candidate),
    'triple_gate_n': donor_n(monocytes, monocytes.obs.triple_gate_candidate)})
fractions['detected_pct_of_monocytes'] = 100 * fractions.SLC7A5_detected_n / fractions.monocyte_n.replace(0, np.nan)
for denom in ['monocyte', 'myeloid', 'total_QC']:
    fractions[f'high_pct_of_{denom}'] = (100 * fractions.high_n / fractions[f'{denom}_n'].replace(0, np.nan)
                                        if high_available else np.nan)
fractions['triple_gate_pct_of_monocytes'] = (100 * fractions.triple_gate_n / fractions.monocyte_n.replace(0, np.nan)
                                            if gate_available else np.nan)
fractions['high_definition_available'] = high_available
fractions['triple_gate_available'] = gate_available
if not high_available:
    fractions['high_n'] = np.nan  # 정의 불가를 0개의 high 집단으로 해석하지 않도록 표시
if not gate_available:
    fractions['triple_gate_n'] = np.nan
fractions.index.name = 'donor' 
fractions.to_csv(ANALYSIS / 'donor_SLC7A5_monocyte_frequencies.csv')
display(fractions)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
ycolumn = 'high_pct_of_monocyte' if high_available else 'detected_pct_of_monocytes'
axes[0].scatter(np.arange(len(DONORS)), fractions[ycolumn], s=65, color='#bd4337')
axes[0].set_xticks(np.arange(len(DONORS)), DONORS)
axes[0].set_ylabel('Monocyte candidates (%)', fontsize=10)
axes[0].set_title('SLC7A5-high candidates' if high_available else 'SLC7A5 detected', fontsize=11)
axes[0].set_ylim(bottom=0)
if monocytes.n_obs == 0:
    axes[0].text(0.5, 0.5, 'No marker-supported\nmonocyte candidates',
                 ha='center', va='center', transform=axes[0].transAxes, fontsize=11)
    axes[0].set_title('Monocyte definition unavailable', fontsize=11)
    axes[0].set_ylabel('')
    axes[0].set_yticks([])
for i, donor in enumerate(DONORS):
    if pd.notna(fractions.loc[donor, ycolumn]):
        numerator = fractions.loc[donor, 'high_n' if high_available else 'SLC7A5_detected_n']
        axes[0].annotate(f'{numerator}/{fractions.loc[donor, "monocyte_n"]}',
                         (i, fractions.loc[donor, ycolumn]), xytext=(0, 8), textcoords='offset points', ha='center', fontsize=8)
composition = pd.crosstab(myeloid.obs.donor, myeloid.obs.subtype_candidate).reindex(DONORS, fill_value=0)
composition.div(composition.sum(axis=1), axis=0).mul(100).plot.bar(stacked=True, ax=axes[1], width=0.8)
axes[1].set_ylabel('Myeloid candidates (%)', fontsize=10)
axes[1].set_title('Candidate subtype composition', fontsize=11)
for axis in axes:
    axis.tick_params(labelsize=10)
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
axes[1].tick_params(axis='x', rotation=0)
fig.tight_layout()
save_current('09_donor_SLC7A5_frequencies_and_myeloid_composition')
composition.to_csv(ANALYSIS / 'donor_myeloid_subtype_counts.csv')

# Descriptive donor-level expression/signature summaries; no cell-level hypothesis testing.
summary = monocytes.obs.groupby(['donor', 'SLC_expression_group'], observed=True).agg(
    n_cells=('sample', 'size'), mean_SLC7A5_log1p=('SLC7A5_log1p', 'mean'),
    mean_reference_signature=('reference_signature_without_SLC7A5', 'mean')).reset_index()
summary.to_csv(ANALYSIS / 'donor_candidate_expression_summary.csv', index=False)
display(summary)
if high_available:
    fig, ax = plt.subplots(figsize=(5.5, 4))
    for donor, group in summary.groupby('donor', observed=True):
        group = group.set_index('SLC_expression_group').reindex(['Other_monocytes', 'SLC7A5_high_candidate'])
        ax.plot([0, 1], group.mean_reference_signature, marker='o', label=str(donor))
    ax.set_xticks([0, 1], ['Other monocytes', 'SLC7A5-high candidate'])
    ax.set_ylabel('Mean reference signature score (PPIF / ANPEP / ITGAX / VCAN)')
    ax.legend(title='Donor', bbox_to_anchor=(1.02, 1))
    fig.tight_layout()
    save_current('10_donor_reference_signature')

## 6.5. MASL 대 MASH: SLC7A5-high monocyte donor-level 비교

실제 `Condition` metadata의 MASL 4명, MASH 4명을 비교합니다. MASL/MASH는 질환군이며 fibrosis stage 값은 아닙니다.
동일한 숫자 suffix만으로 donor를 짝짓지 않습니다. 별도 donor ID 8명을 독립 생물학적 반복으로 취급합니다.

주 지표는 donor별 **SLC7A5-high cluster / 전체 monocyte 비율**입니다.
MASH − MASL의 평균 비율 차이에 대해 **70개 donor label 배치를 모두 계산한 양측 exact permutation test**를 수행합니다.
세포 수로 donor를 가중하지 않으며, pooled Fisher나 cell-level Wilcoxon은 사용하지 않습니다.
집단 선택에는 질환군별 빈도나 P 값을 사용하지 않습니다.

효과 크기는 평균 차이(percentage points)와 각 군 안에서 donor를 재표집한 percentile bootstrap 95% CI입니다.
4명씩의 가능한 bootstrap 조합 65,536개를 전수 계산합니다. 각 군 donor 4명이라는 한계 때문에 CI와 주석은 탐색적으로 해석합니다.
Monocyte 분모 비교 하나가 주 검정이므로 별도 BH 보정은 적용하지 않습니다.
Myeloid/전체 PBMC 분모, expression-q75 gate, donor 하나씩 제외한 평균 차이는 민감도 기술통계로 저장합니다.
포착된 세포 수는 혈액의 절대 세포 수가 아니므로 증가 여부의 주 검정에 사용하지 않습니다.

방법 문서: [independent permutation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.permutation_test.html),
[percentile bootstrap](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html).

**이번 실행의 주석 해석:** 통합 코호트의 primary vs-rest 참조 최상위는 cluster 1(ρ=0.182)이고,
secondary vs-Classical 참조의 최상위는 cluster 0입니다. Cluster 1은 실제 SLC7A5 발현량이 가장 높은
cluster라는 뜻이 아니라, 지정한 primary DEG 참조에 가장 유사한 **잠정적 reference-matched 후보**입니다.
기존 MASL-only cluster 3의 501개 세포 중 378개는 통합 cluster 0, 99개는 통합 cluster 1에 속하므로
이전 cluster 3와 통합 cluster 1을 단순 번호 변경으로 간주하지 않습니다.
Barcode 대응은 `reference_matching/MASL_only_annotation_to_joint_clusters.csv`에 저장했습니다.


In [ ]:
assert DATASET == 'pbmc' and high_available
condition_module_path = HWANG_SCRIPTS / 'HwangBJ_condition_analysis.py'
condition_spec = importlib.util.spec_from_file_location('hwangbj_condition_analysis', condition_module_path)
condition_tools = importlib.util.module_from_spec(condition_spec)
condition_spec.loader.exec_module(condition_tools)
CONDITION_DIR = ANALYSIS / 'condition_comparison'
CONDITION_DIR.mkdir(exist_ok=True)
assert adata.obs.groupby('donor', observed=True).Condition.nunique().eq(1).all()
source_conditions = adata.obs.groupby('donor', observed=True).Condition.first().astype(str)
condition_table = fractions.copy()
condition_table['condition'] = source_conditions.reindex(condition_table.index)
condition_table = condition_table.reset_index()
assert condition_table.groupby('condition').size().to_dict() == {'MASL': 4, 'MASH': 4}
assert np.array_equal(condition_table.high_n.to_numpy(),
    myeloid.obs.loc[myeloid.obs.SLC7A5_reference_cluster, 'donor'].value_counts().reindex(DONORS, fill_value=0).to_numpy())
condition_result, condition_null, condition_loo = condition_tools.compare_donor_conditions(condition_table, seed=SEED)
condition_result.update({'selected_reference_cluster': selected_reference_cluster,
    'reference_Spearman_rho': matched_rho, 'annotation_status': 'reference-matched provisional',
    'cohort_donors': DONORS, 'annotation_selected_using_group_difference': False})
condition_table.to_csv(CONDITION_DIR / 'donor_MASL_MASH_counts_and_frequencies.csv', index=False)
condition_null.to_csv(CONDITION_DIR / 'exact_70_donor_label_permutations.csv', index=False)
condition_loo.to_csv(CONDITION_DIR / 'leave_one_donor_out_mean_differences.csv', index=False)
pd.DataFrame([condition_result]).to_csv(CONDITION_DIR / 'MASL_MASH_primary_comparison.csv', index=False)
(CONDITION_DIR / 'MASL_MASH_primary_comparison.json').write_text(json.dumps(condition_result, indent=2))
condition_means = condition_table.groupby('condition').agg(
    n_donors=('donor', 'size'), mean_pct=('high_pct_of_monocyte', 'mean'),
    median_pct=('high_pct_of_monocyte', 'median'), sd_pct=('high_pct_of_monocyte', 'std'),
    captured_high_cells=('high_n', 'sum'), captured_monocytes=('monocyte_n', 'sum'))
condition_means.to_csv(CONDITION_DIR / 'MASL_MASH_group_descriptive_summary.csv')
# Alternative denominators are descriptive only: no picking the smallest P value.
sensitivity = condition_table[['donor', 'condition', 'high_pct_of_monocyte', 'high_pct_of_myeloid', 'high_pct_of_total_QC']].copy()
q75_counts = monocytes.obs.loc[monocytes.obs.SLC7A5_expression_high_q75, 'donor'].value_counts().reindex(DONORS, fill_value=0)
sensitivity['expression_q75_high_n'] = q75_counts.to_numpy()
sensitivity['expression_q75_pct_of_monocyte'] = 100 * q75_counts.to_numpy() / condition_table.monocyte_n.to_numpy()
sensitivity.to_csv(CONDITION_DIR / 'alternative_denominator_and_expression_gate_descriptive.csv', index=False)
display(condition_table[['donor', 'condition', 'high_n', 'monocyte_n', 'high_pct_of_monocyte']])
display(condition_result)
fig = condition_tools.plot_donor_counts_and_frequencies(condition_table, selected_reference_cluster)
for ext in ['pdf', 'png']:
    fig.savefig(FIGURES / f'16_PBMC_MASL_MASH_donor_counts_and_frequencies.{ext}', dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
fig = condition_tools.plot_condition_comparison(condition_table, condition_result, selected_reference_cluster)
for ext in ['pdf', 'png']:
    fig.savefig(FIGURES / f'17_PBMC_MASL_MASH_SLC7A5high_donor_comparison.{ext}', dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
sc.pl.umap(myeloid, color=['Condition', 'celltype_refined'], ncols=2, show=False)
save_current('18_PBMC_MASL_MASH_reference_annotation_UMAP')
print('MASH mean:', condition_result['MASH_mean_pct'], 'MASL mean:', condition_result['MASL_mean_pct'])
print('MASH increase with two-sided P < 0.05:', condition_result['MASH_increase_P_lt_0_05'])


## 7. 저장 및 후속 donor-level 분석 준비

전체/myeloid/monocyte AnnData와 raw-count pseudobulk를 저장합니다. Pseudobulk는 donor × high/other의 UMI 합입니다.
0-cell 조합은 표에 포함하되 DE 분석용 matrix에서 제외하고, `eligible_for_DE`에 최소 핵 수 조건을 기록합니다.
후속 high-vs-other DE에는 donor를 paired design에 포함하고, 충분한 group별 핵 수가 있는 donor만 사용해야 합니다.
SLC7A5 자체는 그룹 정의에 사용했으므로 독립적인 발견 DEG로 해석하지 않습니다.
MASL 대 MASH 조성 비교는 위 6.5 섹션에서 donor별 비율로 수행합니다. Healthy는 포함되지 않습니다.

In [ ]:
adata.uns['analysis_configuration'] = json.dumps({'dataset': DATASET, 'donors': DONORS,
    'min_genes': MIN_GENES, 'max_mt_pct': MAX_MT_PCT, 'seed': SEED,
    'all_resolution': ALL_RESOLUTION, 'myeloid_resolution': MYELOID_RESOLUTION,
    'batch_correction': 'Harmony', 'batch_key': BATCH_KEY,
    'SLC7A5_annotation_method': 'reference_DEG_correlation',
    'selected_reference_cluster': selected_reference_cluster, 'harmony_theta': HARMONY_THETA, 'harmonypy_version': hm.__version__, 'annotation_status': 'reference-matched provisional'})
coverage = {name: {'present': [g for g in genes if g in adata.var_names],
                   'missing_after_gene_filter': [g for g in genes if g not in adata.var_names]}
            for name, genes in {**LINEAGE, **MYELOID_MARKERS}.items()}
(ANALYSIS / 'marker_panel_coverage.json').write_text(json.dumps(coverage, indent=2))
myeloid.uns['candidate_definition'] = json.dumps(definition)
monocytes.uns['candidate_definition'] = json.dumps(definition)
for obj, filename in [(adata, 'MASL_MASH1_4_all_QC.h5ad'), (myeloid, 'MASL_MASH1_4_myeloid_candidates.h5ad'),
                       (monocytes, 'MASL_MASH1_4_monocyte_candidates.h5ad')]:
    # Marker ranking P values are not retained in delivered analysis objects.
    obj.uns.pop('cluster_marker_ranking', None)
    obj.write_h5ad(ANALYSIS / filename, compression='gzip')

if high_available:
    groups = ['Other_monocytes', 'SLC7A5_high_candidate']
    rows, meta_rows = [], []
    for donor in DONORS:
        for group in groups:
            mask = (monocytes.obs.donor == donor) & (monocytes.obs.SLC_expression_group == group)
            n = int(mask.sum())
            meta_rows.append({'pseudobulk_id': f'{donor}__{group}', 'donor': donor, 'group': group,
                             'n_cells': n, 'eligible_for_DE': n >= MIN_CELLS_PSEUDOBULK,
                             'included_in_matrix': n > 0})
            if n:
                rows.append(sp.csr_matrix(np.asarray(monocytes.layers['counts'][mask.to_numpy()].sum(axis=0), dtype=np.int64)))
    pb_metadata = pd.DataFrame(meta_rows)
    pb_counts = sp.vstack(rows, format='csr')
    sp.save_npz(ANALYSIS / 'monocyte_donor_group_pseudobulk_counts.npz', pb_counts)
    pd.Series(monocytes.var_names, name='gene').to_csv(ANALYSIS / 'pseudobulk_genes.csv', index=False)
    pb_metadata.to_csv(ANALYSIS / 'pseudobulk_metadata_all_combinations.csv', index=False)
    pb_metadata.loc[pb_metadata.included_in_matrix].to_csv(ANALYSIS / 'pseudobulk_matrix_rows.csv', index=False)
    display(pb_metadata)
    assert int(pb_counts.sum()) == int(monocytes.layers['counts'].sum(dtype=np.float64)), 'Pseudobulk count conservation failed'

report = {'dataset': DATASET, 'batch_correction': 'Harmony', 'batch_key': BATCH_KEY,
    'donors': DONORS, 'input_n': int(input_counts.sum()),
    'QC_n': adata.n_obs, 'myeloid_candidate_n': myeloid.n_obs, 'monocyte_candidate_n': monocytes.n_obs,
    'SLC7A5_detected_n': int(monocytes.obs.SLC7A5_detected.sum()) if monocytes.n_obs else None,
    'monocyte_definition_status': 'marker_based_candidate' if monocytes.n_obs else 'no_automatic_marker_supported_candidates',
    'SLC7A5_high_candidate_n': int(monocytes.obs.SLC7A5_high_candidate.sum()) if high_available else None,
    'high_definition_available': bool(high_available), 'annotation_status': 'reference-matched provisional; marker review required',
    'selected_reference_cluster': selected_reference_cluster, 'reference_Spearman_rho': matched_rho,
    'weak_monocyte_marker_clusters': definition['weak_monocyte_marker_clusters'],
    'statistical_testing': condition_result['test'], 'condition_comparison': condition_result,
    'cohort': 'MASL1–4 + MASH1–4, PBMC only'}
(ANALYSIS / 'analysis_summary.json').write_text(json.dumps(report, indent=2))
display(report)
myeloid.write_h5ad(ANALYSIS / 'MASL_MASH1_4_myeloid_reference_annotated.h5ad', compression='gzip')
monocytes.write_h5ad(ANALYSIS / 'MASL_MASH1_4_monocyte_reference_annotated.h5ad', compression='gzip')
print('Analysis:', ANALYSIS)
print('Figures:', FIGURES)

## 두 dataset의 실행 결과 확인

간 핵과 PBMC 결과를 각각 요약합니다. 두 검체의 검출률과 발현 cutoff는 직접적으로 동일하게 해석하지 않습니다.
간 핵은 기존 `harmony/`, MASL/MASH PBMC는 새 `harmony_MASL_MASH/` 폴더에 저장합니다.

In [ ]:
reports = []
for dataset, result_folder in [('liver_nuclei', 'harmony'), ('pbmc', 'harmony_MASL_MASH')]:
    path = PROJECT / '02_Analysis' / 'HwangBJ' / dataset / result_folder / 'analysis_summary.json'
    report_record = json.loads(path.read_text())
    reports.append(report_record)
combined = pd.DataFrame(reports)
combined.to_csv(PROJECT / '02_Analysis' / 'HwangBJ' / 'Harmony_MASL_MASH_dataset_summary.csv', index=False)
display(combined[['dataset', 'batch_correction', 'input_n', 'QC_n', 'myeloid_candidate_n',
                  'monocyte_candidate_n', 'SLC7A5_detected_n', 'SLC7A5_high_candidate_n',
                  'high_definition_available']])


## 결과 검토와 다음 단계

1. 전체 dotplot에서 myeloid 후보가 CSF1R/SPI1/LYZ/LST1 발현을 보이는지, NK/T/hepatocyte contamination이 없는지 확인합니다.
2. Myeloid dotplot과 top-marker CSV로 monocyte/Kupffer/LAM/DC 후보 주석을 검토하고 필요하면 설정 cell의 cluster 목록을 수정합니다.
3. SLC7A5 검출률, high cutoff, high 후보의 donor 기여도를 먼저 확인합니다. 특정 donor에 집중되면 일반화하지 않습니다.
4. PPIF/ANPEP/ITGAX/VCAN, CD14/FCGR3A 및 macrophage/DC marker를 함께 확인한 뒤 기존 Special_monocyte와의 유사성을 판단합니다.
5. 검토가 끝나면 donor를 포함한 paired pseudobulk DE 및 pathway 분석을 별도 추가할 수 있습니다. MASL 대 MASH 비교는 8명의 donor별 비율을 대상으로 exact permutation을 수행합니다. Cell-level 검정을 통한 유의성을 주장하지 않습니다.

Scanpy 문서: [HVG](https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.highly_variable_genes.html),
[marker score](https://scanpy.readthedocs.io/en/stable/generated/scanpy.tl.score_genes.html).